In [0]:
print("Spark version:", spark.version)

print("Databricks connection successful.")

Spark version: 4.1.0
Databricks connection successful.


In [0]:
bronze_path = "abfss://bronze@awdata2022.dfs.core.windows.net/"

display(dbutils.fs.ls(bronze_path))

path,name,size,modificationTime
abfss://bronze@awdata2022.dfs.core.windows.net/production/,production/,0,1795213482000
abfss://bronze@awdata2022.dfs.core.windows.net/sales/,sales/,0,1795133304000


In [0]:
incremental_comparison = (
    small_incremental_df.alias("inc")
    .join(
        customer_full_1.select("CustomerID", "ModifiedDate").alias("base"),
        on="CustomerID",
        how="left"
    )
    .select(
        "CustomerID",
        F.col("inc.ModifiedDate").alias("IncrementalModifiedDate"),
        F.col("base.ModifiedDate").alias("BaselineModifiedDate"),
        F.when(
            F.col("base.CustomerID").isNull(),
            "New customer"
        ).when(
            F.col("inc.ModifiedDate") > F.col("base.ModifiedDate"),
            "Updated customer"
        ).otherwise(
            "Same or older version"
        ).alias("ChangeType")
    )
    .orderBy("CustomerID")
)

display(incremental_comparison)

CustomerID,IncrementalModifiedDate,BaselineModifiedDate,ChangeType
1,2014-09-13T00:00:00Z,2014-09-12T11:15:07.263Z,Updated customer
2,2014-09-14T10:00:00Z,2014-09-12T11:15:07.263Z,Updated customer
2,2014-09-14T10:00:00Z,2014-09-12T11:15:07.263Z,Updated customer
3,2014-09-14T10:00:00Z,2014-09-12T11:15:07.263Z,Updated customer
3,2014-09-14T10:00:00Z,2014-09-12T11:15:07.263Z,Updated customer
4,2014-09-14T10:00:00Z,2014-09-12T11:15:07.263Z,Updated customer
4,2014-09-14T10:00:00Z,2014-09-12T11:15:07.263Z,Updated customer


In [0]:
from pyspark.sql.window import Window
from pyspark.sql import functions as F

customer_window = (
    Window
    .partitionBy("CustomerID")
    .orderBy(F.col("ModifiedDate").desc())
)

deduplicated_incremental = (
    small_incremental_df
    .withColumn(
        "row_number",
        F.row_number().over(customer_window)
    )
    .filter(F.col("row_number") == 1)
    .drop("row_number")
)

print("Original incremental rows:", small_incremental_df.count())
print("Rows after deduplication:", deduplicated_incremental.count())

display(
    deduplicated_incremental
    .select("CustomerID", "AccountNumber", "ModifiedDate")
    .orderBy("CustomerID")
)

Original incremental rows: 7
Rows after deduplication: 4


CustomerID,AccountNumber,ModifiedDate
1,AW00000001,2014-09-13T00:00:00Z
2,AW00000002,2014-09-14T10:00:00Z
3,AW00000003,2014-09-14T10:00:00Z
4,AW00000004,2014-09-14T10:00:00Z


In [0]:
from pyspark.sql.window import Window
from pyspark.sql import functions as F

merge_window = (
    Window
    .partitionBy("CustomerID")
    .orderBy(F.col("ModifiedDate").desc())
)

customer_latest = (
    customer_combined
    .withColumn(
        "row_number",
        F.row_number().over(merge_window)
    )
    .filter(F.col("row_number") == 1)
    .drop("row_number")
)

print("Rows after deduplication:", customer_latest.count())

print(
    "Distinct CustomerIDs:",
    customer_latest.select("CustomerID").distinct().count()
)

display(
    customer_latest
    .filter(F.col("CustomerID").isin(1, 2, 3, 4))
    .select("CustomerID", "AccountNumber", "ModifiedDate")
    .orderBy("CustomerID")
)

Rows after deduplication: 19820
Distinct CustomerIDs: 19820


CustomerID,AccountNumber,ModifiedDate
1,AW00000001,2014-09-13T00:00:00Z
2,AW00000002,2014-09-14T10:00:00Z
3,AW00000003,2014-09-14T10:00:00Z
4,AW00000004,2014-09-14T10:00:00Z


In [0]:
print("Total Silver-ready rows:", customer_latest.count())

print(
    "Distinct CustomerIDs:",
    customer_latest.select("CustomerID").distinct().count()
)

print(
    "Duplicate CustomerIDs:",
    customer_latest.groupBy("CustomerID")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

Total Silver-ready rows: 19820
Distinct CustomerIDs: 19820
Duplicate CustomerIDs: 0


In [0]:
print("Current catalog:", spark.catalog.currentCatalog())
print("Current schema:", spark.catalog.currentDatabase())

display(spark.sql("SHOW CATALOGS"))

Current catalog: adb_adventureworks_dev
Current schema: default


catalog
adb_adventureworks_dev
samples
system


In [0]:
spark.sql("""
CREATE SCHEMA IF NOT EXISTS adb_adventureworks_dev.silver
COMMENT 'Cleaned and validated Silver-layer tables for the AdventureWorks data platform'
""")

print("Silver schema created successfully.")

Silver schema created successfully.


In [0]:
display(
    spark.sql(
        "SHOW SCHEMAS IN adb_adventureworks_dev"
    )
)

databaseName
default
information_schema
silver


In [0]:
display(
    spark.sql(
        "SHOW TABLES IN adb_adventureworks_dev.silver"
    )
)

database,tableName,isTemporary


In [0]:
silver_table = "adb_adventureworks_dev.silver.customer"

customer_latest.createOrReplaceTempView("customer_silver_stage")

spark.sql("""
CREATE TABLE adb_adventureworks_dev.silver.customer
USING DELTA
AS
SELECT *
FROM customer_silver_stage
""")

print(f"Created Silver table: {silver_table}")

---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-5893516122533481>, line 3
      1 silver_table = "adb_adventureworks_dev.silver.customer"
----> 3 customer_latest.createOrReplaceTempView("customer_silver_stage")
      5 spark.sql("""
      6 CREATE TABLE adb_adventureworks_dev.silver.customer
      7 USING DELTA
   (...)
     10 FROM customer_silver_stage
     11 """)
     13 print(f"Created Silver table: {silver_table}")

NameError: name 'customer_latest' is not defined

In [0]:
silver_customer = spark.table(
    "adb_adventureworks_dev.silver.customer"
)

print("Total rows:", silver_customer.count())

print(
    "Distinct CustomerIDs:",
    silver_customer.select("CustomerID").distinct().count()
)

print(
    "Duplicate CustomerIDs:",
    silver_customer.groupBy("CustomerID")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

display(silver_customer.limit(10))

Total rows: 19820
Distinct CustomerIDs: 19820
Duplicate CustomerIDs: 0


CustomerID,PersonID,StoreID,TerritoryID,AccountNumber,rowguid,ModifiedDate
1,null,934,1,AW00000001,3f5ae95e-b87d-4aed-95b4-c3797afcb74f,2014-09-13T00:00:00Z
2,null,1028,1,AW00000002,e552f657-a9af-4a7d-a645-c429d6e02491,2014-09-14T10:00:00Z
3,null,642,4,AW00000003,130774b1-db21-4ef3-98c8-c104bcd6ed6d,2014-09-14T10:00:00Z
4,null,932,4,AW00000004,ff862851-1daa-4044-be7c-3e85583c054d,2014-09-14T10:00:00Z
5,null,1026,4,AW00000005,83905bdc-6f5e-4f71-b162-c98da069f38a,2014-09-12T11:15:07.263Z
6,null,644,4,AW00000006,1a92df88-bfa2-467d-bd54-fcb9e647fdd7,2014-09-12T11:15:07.263Z
7,null,930,1,AW00000007,03e9273e-b193-448e-9823-fe0c44aeed78,2014-09-12T11:15:07.263Z
8,null,1024,5,AW00000008,801368b1-4323-4bfa-8bea-5b5b1e4bd4a0,2014-09-12T11:15:07.263Z
9,null,620,5,AW00000009,b900bb7f-23c3-481d-80da-c49a5bd6f772,2014-09-12T11:15:07.263Z
10,null,928,6,AW00000010,cdb6698d-2ff1-4fba-8f22-60ad1d11dabd,2014-09-12T11:15:07.263Z


In [0]:
customer_type_check = silver_customer.agg(
    F.count("*").alias("TotalCustomers"),

    F.sum(
        F.when(
            F.col("PersonID").isNotNull()
            & F.col("StoreID").isNull(), 1
        ).otherwise(0)
    ).alias("IndividualCustomers"),

    F.sum(
        F.when(
            F.col("PersonID").isNull()
            & F.col("StoreID").isNotNull(), 1
        ).otherwise(0)
    ).alias("StoreCustomers"),

    F.sum(
        F.when(
            F.col("PersonID").isNull()
            & F.col("StoreID").isNull(), 1
        ).otherwise(0)
    ).alias("MissingBoth"),

    F.sum(
        F.when(
            F.col("PersonID").isNotNull()
            & F.col("StoreID").isNotNull(), 1
        ).otherwise(0)
    ).alias("PopulatedBoth")
)

display(customer_type_check)

TotalCustomers,IndividualCustomers,StoreCustomers,MissingBoth,PopulatedBoth
19820,18484,701,0,635


In [0]:
display(
    silver_customer
    .filter(
        F.col("PersonID").isNotNull()
        & F.col("StoreID").isNotNull()
    )
    .select(
        "CustomerID",
        "PersonID",
        "StoreID",
        "TerritoryID",
        "AccountNumber"
    )
    .orderBy("CustomerID")
    .limit(15)
)

CustomerID,PersonID,StoreID,TerritoryID,AccountNumber
29484,291,292,5,AW00029484
29485,293,294,4,AW00029485
29486,295,296,3,AW00029486
29487,297,298,2,AW00029487
29488,299,300,9,AW00029488
29489,301,302,4,AW00029489
29490,303,304,1,AW00029490
29491,305,306,5,AW00029491
29492,307,308,3,AW00029492
29493,309,310,5,AW00029493


In [0]:
display(
    spark.sql("""
        DESCRIBE HISTORY adb_adventureworks_dev.silver.customer
    """)
)

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
0,2026-11-20T21:26:23.197Z,142540505220658,lehlogonoloramphisa@outlook.com,CREATE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""databricks.internal.autoUpgrades.delta.enableRowTracking"":""true"",""delta.enableDeletionVectors"":""true"",""databricks.internal.autoUpgrades.delta.checkpointPolicy"":""v2"",""databricks.internal.autoUpgrades.delta.parquet.format.version"":""2.12.0"",""delta.parquet.format.version"":""2.12.0"",""delta.enableRowTracking"":""true"",""delta.checkpointPolicy"":""v2"",""io.unitycatalog.tableId"":""30295282-2599-46eb-bf3b-6ef3840738d4"",""delta.rowTracking.materializedRowCommitVersionColumnName"":""_row-commit-version-col-2b552177-ac94-47ef-9099-539b66564a0d"",""delta.enableInCommitTimestamps"":""true"",""databricks.internal.autoUpgrades.delta.enableInCommitTimestamps"":""true"",""delta.rowTracking.materializedRowIdColumnName"":""_row-id-col-6ac2984b-529c-41eb-8455-69f6a4d44262""}, statsOnLoad -> true, clusterByAuto -> true, clusteringOnWriteStatus -> Reason for skipping: Nonexistent clustering column, generally when replacing a Liquid table with a table without clustering columns)",null,List(1361742418849894),null,1120-175118-j84pd9yh,null,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 19820, numOutputBytes -> 523626)",null,Databricks-Runtime/18.x-photon-scala2.13


In [0]:
salesorderheader_path = (
    bronze_path + "sales/salesorderheader/"
)

display(
    dbutils.fs.ls(salesorderheader_path)
)

path,name,size,modificationTime
abfss://bronze@awdata2022.dfs.core.windows.net/sales/salesorderheader/SalesOrderHeader.parquet,SalesOrderHeader.parquet,3030945,1795136670000
abfss://bronze@awdata2022.dfs.core.windows.net/sales/salesorderheader/incremental/,incremental/,0,1795140326000


In [0]:
salesorderheader_incremental_path = (
    salesorderheader_path + "incremental/"
)

display(
    dbutils.fs.ls(salesorderheader_incremental_path)
)

path,name,size,modificationTime
abfss://bronze@awdata2022.dfs.core.windows.net/sales/salesorderheader/incremental/SalesOrderHeader_2b1d3b82-c0d2-4433-8466-42d121c4c664.parquet,SalesOrderHeader_2b1d3b82-c0d2-4433-8466-42d121c4c664.parquet,9449,1795201315000
abfss://bronze@awdata2022.dfs.core.windows.net/sales/salesorderheader/incremental/SalesOrderHeader_35f97c3e-cffc-40d0-862e-b0685d6c35e1.parquet,SalesOrderHeader_35f97c3e-cffc-40d0-862e-b0685d6c35e1.parquet,9449,1795206444000
abfss://bronze@awdata2022.dfs.core.windows.net/sales/salesorderheader/incremental/SalesOrderHeader_4d2e4808-6497-4eb1-8a6c-59088fe08359.parquet,SalesOrderHeader_4d2e4808-6497-4eb1-8a6c-59088fe08359.parquet,9449,1795207081000
abfss://bronze@awdata2022.dfs.core.windows.net/sales/salesorderheader/incremental/SalesOrderHeader_77efa261-e26e-4ce8-acb0-034232871915.parquet,SalesOrderHeader_77efa261-e26e-4ce8-acb0-034232871915.parquet,9449,1795141635000
abfss://bronze@awdata2022.dfs.core.windows.net/sales/salesorderheader/incremental/SalesOrderHeader_7d981eec-44ea-4bf7-97c3-f76819c11244.parquet,SalesOrderHeader_7d981eec-44ea-4bf7-97c3-f76819c11244.parquet,9449,1795140327000
abfss://bronze@awdata2022.dfs.core.windows.net/sales/salesorderheader/incremental/SalesOrderHeader_c931e768-2d15-43b3-8ccd-c7c7902c5d45.parquet,SalesOrderHeader_c931e768-2d15-43b3-8ccd-c7c7902c5d45.parquet,9449,1795142064000


In [0]:
from pyspark.sql import functions as F

salesorderheader_incremental_df = spark.read.parquet(
    salesorderheader_incremental_path
)

header_file_summary = (
    salesorderheader_incremental_df
    .withColumn(
        "SourceFile",
        F.regexp_extract(
            F.col("_metadata.file_path"),
            r"([^/]+)$",
            1
        )
    )
    .groupBy("SourceFile")
    .agg(
        F.count("*").alias("RowCount"),
        F.min("ModifiedDate").alias("OldestModifiedDate"),
        F.max("ModifiedDate").alias("LatestModifiedDate")
    )
    .orderBy("SourceFile")
)

display(header_file_summary)

SourceFile,RowCount,OldestModifiedDate,LatestModifiedDate
SalesOrderHeader_2b1d3b82-c0d2-4433-8466-42d121c4c664.parquet,40,2014-07-07T00:00:00Z,2014-07-07T00:00:00Z
SalesOrderHeader_35f97c3e-cffc-40d0-862e-b0685d6c35e1.parquet,40,2014-07-07T00:00:00Z,2014-07-07T00:00:00Z
SalesOrderHeader_4d2e4808-6497-4eb1-8a6c-59088fe08359.parquet,40,2014-07-07T00:00:00Z,2014-07-07T00:00:00Z
SalesOrderHeader_77efa261-e26e-4ce8-acb0-034232871915.parquet,40,2014-07-07T00:00:00Z,2014-07-07T00:00:00Z
SalesOrderHeader_7d981eec-44ea-4bf7-97c3-f76819c11244.parquet,40,2014-07-07T00:00:00Z,2014-07-07T00:00:00Z
SalesOrderHeader_c931e768-2d15-43b3-8ccd-c7c7902c5d45.parquet,40,2014-07-07T00:00:00Z,2014-07-07T00:00:00Z


In [0]:
from pyspark.sql import functions as F

salesorderheader_incremental_df = spark.read.parquet(
    salesorderheader_incremental_path
)

incremental_check = salesorderheader_incremental_df.agg(
    F.count("*").alias("TotalRows"),
    F.countDistinct("SalesOrderID").alias("DistinctSalesOrderIDs"),
    F.countDistinct(
        F.struct(*salesorderheader_incremental_df.columns)
    ).alias("DistinctFullRecords"),
    F.min("ModifiedDate").alias("OldestModifiedDate"),
    F.max("ModifiedDate").alias("LatestModifiedDate")
)

display(incremental_check)

TotalRows,DistinctSalesOrderIDs,DistinctFullRecords,OldestModifiedDate,LatestModifiedDate
240,40,40,2014-07-07T00:00:00Z,2014-07-07T00:00:00Z


In [0]:
salesorderheader_full_path = salesorderheader_path + "SalesOrderHeader.parquet"

salesorderheader_full_df = spark.read.parquet(
    salesorderheader_full_path
)

salesorderheader_full_df.printSchema()

header_baseline_check = salesorderheader_full_df.agg(
    F.count("*").alias("TotalRows"),
    F.countDistinct("SalesOrderID").alias("DistinctSalesOrderIDs"),
    F.sum(
        F.when(F.col("SalesOrderID").isNull(), 1).otherwise(0)
    ).alias("NullSalesOrderIDs"),
    F.min("ModifiedDate").alias("OldestModifiedDate"),
    F.max("ModifiedDate").alias("LatestModifiedDate")
)

display(header_baseline_check)


root
 |-- SalesOrderID: integer (nullable = true)
 |-- RevisionNumber: integer (nullable = true)
 |-- OrderDate: timestamp (nullable = true)
 |-- DueDate: timestamp (nullable = true)
 |-- ShipDate: timestamp (nullable = true)
 |-- Status: integer (nullable = true)
 |-- OnlineOrderFlag: boolean (nullable = true)
 |-- SalesOrderNumber: string (nullable = true)
 |-- PurchaseOrderNumber: string (nullable = true)
 |-- AccountNumber: string (nullable = true)
 |-- CustomerID: integer (nullable = true)
 |-- SalesPersonID: integer (nullable = true)
 |-- TerritoryID: integer (nullable = true)
 |-- BillToAddressID: integer (nullable = true)
 |-- ShipToAddressID: integer (nullable = true)
 |-- ShipMethodID: integer (nullable = true)
 |-- CreditCardID: integer (nullable = true)
 |-- CreditCardApprovalCode: string (nullable = true)
 |-- CurrencyRateID: integer (nullable = true)
 |-- SubTotal: decimal(19,4) (nullable = true)
 |-- TaxAmt: decimal(19,4) (nullable = true)
 |-- Freight: decimal(19,4) (nu

TotalRows,DistinctSalesOrderIDs,NullSalesOrderIDs,OldestModifiedDate,LatestModifiedDate
31465,31465,0,2011-06-07T00:00:00Z,2014-07-07T00:00:00Z


In [0]:
display(header_baseline_check)

TotalRows,DistinctSalesOrderIDs,NullSalesOrderIDs,OldestModifiedDate,LatestModifiedDate
31465,31465,0,2011-06-07T00:00:00Z,2014-07-07T00:00:00Z


In [0]:
# Remove exact duplicates from the incremental extracts.
header_incremental_distinct = (
    salesorderheader_incremental_df
    .dropDuplicates()
)

# Get the matching records from the full-load baseline.
header_baseline_matches = (
    salesorderheader_full_df
    .join(
        header_incremental_distinct.select("SalesOrderID"),
        on="SalesOrderID",
        how="inner"
    )
    .select(*salesorderheader_full_df.columns)
)

# Compare every column in both directions.
incremental_not_in_baseline = (
    header_incremental_distinct
    .exceptAll(header_baseline_matches)
)

baseline_not_in_incremental = (
    header_baseline_matches
    .exceptAll(header_incremental_distinct)
)

comparison_results = [
    ("Distinct incremental records", header_incremental_distinct.count()),
    ("Matching baseline records", header_baseline_matches.count()),
    ("Incremental records differing from baseline",
     incremental_not_in_baseline.count()),
    ("Baseline records differing from incremental",
     baseline_not_in_incremental.count())
]

display(
    spark.createDataFrame(
        comparison_results,
        ["Check", "RowCount"]
    )
)

Check,RowCount
Distinct incremental records,40
Matching baseline records,40
Incremental records differing from baseline,0
Baseline records differing from incremental,0


In [0]:
from pyspark.sql import functions as F

header_quality_check = salesorderheader_full_df.agg(
    F.sum(F.when(F.col("SalesOrderNumber").isNull(), 1).otherwise(0)).alias("NullSalesOrderNumber"),
    F.sum(F.when(F.col("OrderDate").isNull(), 1).otherwise(0)).alias("NullOrderDate"),
    F.sum(F.when(F.col("CustomerID").isNull(), 1).otherwise(0)).alias("NullCustomerID"),
    F.sum(F.when(F.col("TotalDue").isNull(), 1).otherwise(0)).alias("NullTotalDue"),
    F.sum(F.when(F.col("Status").isNull(), 1).otherwise(0)).alias("NullStatus"),
    F.sum(F.when(F.col("SalesOrderID") <= 0, 1).otherwise(0)).alias("InvalidSalesOrderID"),
    F.sum(F.when(~F.col("Status").between(1, 6), 1).otherwise(0)).alias("InvalidStatus"),
    F.sum(F.when(F.col("DueDate") < F.col("OrderDate"), 1).otherwise(0)).alias("DueDateBeforeOrderDate"),
    F.sum(
        F.when(
            F.abs(
                F.col("TotalDue") -
                (F.col("SubTotal") + F.col("TaxAmt") + F.col("Freight"))
            ) > F.lit(0.0001),
            1
        ).otherwise(0)
    ).alias("MonetaryTotalMismatches")
)

display(header_quality_check)

NullSalesOrderNumber,NullOrderDate,NullCustomerID,NullTotalDue,NullStatus,InvalidSalesOrderID,InvalidStatus,DueDateBeforeOrderDate,MonetaryTotalMismatches
0,0,0,0,0,0,0,0,0


In [0]:
silver_customer = spark.table(
    "adb_adventureworks_dev.silver.customer"
)

header_customer_check = (
    salesorderheader_full_df
    .join(
        silver_customer.select("CustomerID"),
        on="CustomerID",
        how="left_anti"
    )
)

display(
    header_customer_check.select(
        "SalesOrderID",
        "CustomerID"
    )
)

print(
    "Orders with no matching Silver customer:",
    header_customer_check.count()
)

SalesOrderID,CustomerID


Orders with no matching Silver customer: 0


In [0]:
salesorderheader_full_df.createOrReplaceTempView(
    "salesorderheader_silver_stage"
)

spark.sql("""
CREATE TABLE adb_adventureworks_dev.silver.salesorderheader
USING DELTA
AS
SELECT *
FROM salesorderheader_silver_stage
""")

DataFrame[num_affected_rows: bigint, num_inserted_rows: bigint]

In [0]:
silver_salesorderheader = spark.table(
    "adb_adventureworks_dev.silver.salesorderheader"
)

header_silver_verification = silver_salesorderheader.agg(
    F.count("*").alias("TotalRows"),
    F.countDistinct("SalesOrderID").alias("DistinctSalesOrderIDs"),
    F.sum(
        F.when(F.col("SalesOrderID").isNull(), 1).otherwise(0)
    ).alias("NullSalesOrderIDs"),
    F.min("ModifiedDate").alias("OldestModifiedDate"),
    F.max("ModifiedDate").alias("LatestModifiedDate")
)

display(header_silver_verification)

TotalRows,DistinctSalesOrderIDs,NullSalesOrderIDs,OldestModifiedDate,LatestModifiedDate
31465,31465,0,2011-06-07T00:00:00Z,2014-07-07T00:00:00Z


In [0]:
salesorderdetail_path = (
    bronze_path + "sales/salesorderdetail/"
)

display(dbutils.fs.ls(salesorderdetail_path))

path,name,size,modificationTime
abfss://bronze@awdata2022.dfs.core.windows.net/sales/salesorderdetail/SalesOrderDetail.parquet,SalesOrderDetail.parquet,6275757,1795136651000
abfss://bronze@awdata2022.dfs.core.windows.net/sales/salesorderdetail/incremental/,incremental/,0,1795140312000


In [0]:
salesorderdetail_incremental_path = (
    salesorderdetail_path + "incremental/"
)

display(dbutils.fs.ls(salesorderdetail_incremental_path))

path,name,size,modificationTime
abfss://bronze@awdata2022.dfs.core.windows.net/sales/salesorderdetail/incremental/SalesOrderDetail_2b1d3b82-c0d2-4433-8466-42d121c4c664.parquet,SalesOrderDetail_2b1d3b82-c0d2-4433-8466-42d121c4c664.parquet,7635,1795201283000
abfss://bronze@awdata2022.dfs.core.windows.net/sales/salesorderdetail/incremental/SalesOrderDetail_35f97c3e-cffc-40d0-862e-b0685d6c35e1.parquet,SalesOrderDetail_35f97c3e-cffc-40d0-862e-b0685d6c35e1.parquet,7635,1795206421000
abfss://bronze@awdata2022.dfs.core.windows.net/sales/salesorderdetail/incremental/SalesOrderDetail_4d2e4808-6497-4eb1-8a6c-59088fe08359.parquet,SalesOrderDetail_4d2e4808-6497-4eb1-8a6c-59088fe08359.parquet,7635,1795207054000
abfss://bronze@awdata2022.dfs.core.windows.net/sales/salesorderdetail/incremental/SalesOrderDetail_77efa261-e26e-4ce8-acb0-034232871915.parquet,SalesOrderDetail_77efa261-e26e-4ce8-acb0-034232871915.parquet,7635,1795141608000
abfss://bronze@awdata2022.dfs.core.windows.net/sales/salesorderdetail/incremental/SalesOrderDetail_7d981eec-44ea-4bf7-97c3-f76819c11244.parquet,SalesOrderDetail_7d981eec-44ea-4bf7-97c3-f76819c11244.parquet,7635,1795140312000
abfss://bronze@awdata2022.dfs.core.windows.net/sales/salesorderdetail/incremental/SalesOrderDetail_c931e768-2d15-43b3-8ccd-c7c7902c5d45.parquet,SalesOrderDetail_c931e768-2d15-43b3-8ccd-c7c7902c5d45.parquet,7635,1795142031000


In [0]:
from pyspark.sql import functions as F

salesorderdetail_incremental_df = spark.read.parquet(
    salesorderdetail_incremental_path
)

detail_file_summary = (
    salesorderdetail_incremental_df
    .withColumn(
        "SourceFile",
        F.regexp_extract(
            F.col("_metadata.file_path"),
            r"([^/]+)$",
            1
        )
    )
    .groupBy("SourceFile")
    .agg(
        F.count("*").alias("RowCount"),
        F.min("ModifiedDate").alias("OldestModifiedDate"),
        F.max("ModifiedDate").alias("LatestModifiedDate")
    )
    .orderBy("SourceFile")
)

display(detail_file_summary)

SourceFile,RowCount,OldestModifiedDate,LatestModifiedDate
SalesOrderDetail_2b1d3b82-c0d2-4433-8466-42d121c4c664.parquet,96,2014-06-30T00:00:00Z,2014-06-30T00:00:00Z
SalesOrderDetail_35f97c3e-cffc-40d0-862e-b0685d6c35e1.parquet,96,2014-06-30T00:00:00Z,2014-06-30T00:00:00Z
SalesOrderDetail_4d2e4808-6497-4eb1-8a6c-59088fe08359.parquet,96,2014-06-30T00:00:00Z,2014-06-30T00:00:00Z
SalesOrderDetail_77efa261-e26e-4ce8-acb0-034232871915.parquet,96,2014-06-30T00:00:00Z,2014-06-30T00:00:00Z
SalesOrderDetail_7d981eec-44ea-4bf7-97c3-f76819c11244.parquet,96,2014-06-30T00:00:00Z,2014-06-30T00:00:00Z
SalesOrderDetail_c931e768-2d15-43b3-8ccd-c7c7902c5d45.parquet,96,2014-06-30T00:00:00Z,2014-06-30T00:00:00Z


In [0]:
detail_incremental_check = salesorderdetail_incremental_df.agg(
    F.count("*").alias("TotalRows"),
    F.countDistinct(
        F.struct("SalesOrderID", "SalesOrderDetailID")
    ).alias("DistinctCompositeKeys"),
    F.countDistinct(
        F.struct(*salesorderdetail_incremental_df.columns)
    ).alias("DistinctFullRecords"),
    F.sum(
        F.when(
            F.col("SalesOrderID").isNull() |
            F.col("SalesOrderDetailID").isNull(),
            1
        ).otherwise(0)
    ).alias("RowsWithNullKeyFields")
)

display(detail_incremental_check)

TotalRows,DistinctCompositeKeys,DistinctFullRecords,RowsWithNullKeyFields
576,96,96,0


In [0]:
salesorderdetail_full_path = (
    salesorderdetail_path + "SalesOrderDetail.parquet"
)

salesorderdetail_full_df = spark.read.parquet(
    salesorderdetail_full_path
)

salesorderdetail_full_df.printSchema()

detail_baseline_check = salesorderdetail_full_df.agg(
    F.count("*").alias("TotalRows"),
    F.countDistinct(
        F.struct("SalesOrderID", "SalesOrderDetailID")
    ).alias("DistinctCompositeKeys"),
    F.sum(
        F.when(
            F.col("SalesOrderID").isNull() |
            F.col("SalesOrderDetailID").isNull(),
            1
        ).otherwise(0)
    ).alias("RowsWithNullKeyFields"),
    F.min("ModifiedDate").alias("OldestModifiedDate"),
    F.max("ModifiedDate").alias("LatestModifiedDate")
)

display(detail_baseline_check)

root
 |-- SalesOrderID: integer (nullable = true)
 |-- SalesOrderDetailID: integer (nullable = true)
 |-- CarrierTrackingNumber: string (nullable = true)
 |-- OrderQty: integer (nullable = true)
 |-- ProductID: integer (nullable = true)
 |-- SpecialOfferID: integer (nullable = true)
 |-- UnitPrice: decimal(19,4) (nullable = true)
 |-- UnitPriceDiscount: decimal(19,4) (nullable = true)
 |-- LineTotal: decimal(38,6) (nullable = true)
 |-- rowguid: string (nullable = true)
 |-- ModifiedDate: timestamp (nullable = true)



TotalRows,DistinctCompositeKeys,RowsWithNullKeyFields,OldestModifiedDate,LatestModifiedDate
121317,121317,0,2011-05-31T00:00:00Z,2014-06-30T00:00:00Z


In [0]:
# Remove repeated copies across the six incremental files.
detail_incremental_distinct = (
    salesorderdetail_incremental_df
    .dropDuplicates()
)

# Find the matching composite keys in the full-load baseline.
detail_baseline_matches = (
    salesorderdetail_full_df
    .join(
        detail_incremental_distinct.select(
            "SalesOrderID",
            "SalesOrderDetailID"
        ),
        on=["SalesOrderID", "SalesOrderDetailID"],
        how="inner"
    )
    .select(*salesorderdetail_full_df.columns)
)

# Compare all columns in both directions.
detail_incremental_not_in_baseline = (
    detail_incremental_distinct
    .exceptAll(detail_baseline_matches)
)

detail_baseline_not_in_incremental = (
    detail_baseline_matches
    .exceptAll(detail_incremental_distinct)
)

detail_comparison_results = [
    ("Distinct incremental records", detail_incremental_distinct.count()),
    ("Matching baseline records", detail_baseline_matches.count()),
    (
        "Incremental records differing from baseline",
        detail_incremental_not_in_baseline.count()
    ),
    (
        "Baseline records differing from incremental",
        detail_baseline_not_in_incremental.count()
    )
]

display(
    spark.createDataFrame(
        detail_comparison_results,
        ["Check", "RowCount"]
    )
)

Check,RowCount
Distinct incremental records,96
Matching baseline records,96
Incremental records differing from baseline,0
Baseline records differing from incremental,0


In [0]:
detail_quality_check = salesorderdetail_full_df.agg(
    F.count("*").alias("TotalRows"),

    F.sum(F.when(F.col("OrderQty").isNull(), 1).otherwise(0))
        .alias("NullOrderQty"),

    F.sum(F.when(F.col("ProductID").isNull(), 1).otherwise(0))
        .alias("NullProductID"),

    F.sum(F.when(F.col("UnitPrice").isNull(), 1).otherwise(0))
        .alias("NullUnitPrice"),

    F.sum(F.when(F.col("UnitPriceDiscount").isNull(), 1).otherwise(0))
        .alias("NullUnitPriceDiscount"),

    F.sum(F.when(F.col("LineTotal").isNull(), 1).otherwise(0))
        .alias("NullLineTotal"),

    F.sum(F.when(F.col("OrderQty") <= 0, 1).otherwise(0))
        .alias("InvalidOrderQty"),

    F.sum(F.when(F.col("ProductID") <= 0, 1).otherwise(0))
        .alias("InvalidProductID"),

    F.sum(F.when(F.col("UnitPrice") < 0, 1).otherwise(0))
        .alias("NegativeUnitPrice"),

    F.sum(
        F.when(
            (F.col("UnitPriceDiscount") < 0) |
            (F.col("UnitPriceDiscount") > 1),
            1
        ).otherwise(0)
    ).alias("InvalidDiscount"),

    F.sum(
        F.when(
            F.abs(
                F.col("LineTotal") -
                (
                    F.col("UnitPrice") *
                    (1 - F.col("UnitPriceDiscount")) *
                    F.col("OrderQty")
                )
            ) > F.lit(0.000001),
            1
        ).otherwise(0)
    ).alias("LineTotalMismatches")
)

display(detail_quality_check)

TotalRows,NullOrderQty,NullProductID,NullUnitPrice,NullUnitPriceDiscount,NullLineTotal,InvalidOrderQty,InvalidProductID,NegativeUnitPrice,InvalidDiscount,LineTotalMismatches
121317,0,0,0,0,0,0,0,0,0,0


In [0]:
detail_order_check = (
    salesorderdetail_full_df
    .join(
        silver_salesorderheader.select("SalesOrderID"),
        on="SalesOrderID",
        how="left_anti"
    )
)

display(
    detail_order_check.select(
        "SalesOrderID",
        "SalesOrderDetailID"
    )
)

print(
    "Detail lines without a matching sales order:",
    detail_order_check.count()
)

SalesOrderID,SalesOrderDetailID


Detail lines without a matching sales order: 0


In [0]:
salesorderdetail_full_df.createOrReplaceTempView(
    "salesorderdetail_silver_stage"
)

spark.sql("""
CREATE TABLE adb_adventureworks_dev.silver.salesorderdetail
USING DELTA
AS
SELECT *
FROM salesorderdetail_silver_stage
""")

DataFrame[num_affected_rows: bigint, num_inserted_rows: bigint]

In [0]:
silver_salesorderdetail = spark.table(
    "adb_adventureworks_dev.silver.salesorderdetail"
)

detail_silver_verification = silver_salesorderdetail.agg(
    F.count("*").alias("TotalRows"),

    F.countDistinct(
        F.struct("SalesOrderID", "SalesOrderDetailID")
    ).alias("DistinctCompositeKeys"),

    F.sum(
        F.when(
            F.col("SalesOrderID").isNull() |
            F.col("SalesOrderDetailID").isNull(),
            1
        ).otherwise(0)
    ).alias("RowsWithNullKeyFields"),

    F.min("ModifiedDate").alias("OldestModifiedDate"),
    F.max("ModifiedDate").alias("LatestModifiedDate")
)

display(detail_silver_verification)

TotalRows,DistinctCompositeKeys,RowsWithNullKeyFields,OldestModifiedDate,LatestModifiedDate
121317,121317,0,2011-05-31T00:00:00Z,2014-06-30T00:00:00Z


In [0]:
silver_tables = [
    "adb_adventureworks_dev.silver.customer",
    "adb_adventureworks_dev.silver.salesorderheader",
    "adb_adventureworks_dev.silver.salesorderdetail"
]

for table_name in silver_tables:
    print(f"\nTable: {table_name}")

    table_df = spark.table(table_name)

    print("Rows:", table_df.count())
    print("Columns:", len(table_df.columns))

    display(
        spark.sql(f"DESCRIBE HISTORY {table_name}")
        .select("version", "timestamp", "operation", "operationMetrics")
        .limit(1)
    )


Table: adb_adventureworks_dev.silver.customer
Rows: 19820
Columns: 7


version,timestamp,operation,operationMetrics
0,2026-11-20T21:26:23.197Z,CREATE TABLE AS SELECT,"Map(numFiles -> 1, numOutputRows -> 19820, numOutputBytes -> 523626)"



Table: adb_adventureworks_dev.silver.salesorderheader
Rows: 31465
Columns: 26


version,timestamp,operation,operationMetrics
0,2026-11-20T21:48:35.826Z,CREATE TABLE AS SELECT,"Map(numFiles -> 1, numOutputRows -> 31465, numOutputBytes -> 1713024)"



Table: adb_adventureworks_dev.silver.salesorderdetail
Rows: 121317
Columns: 11


version,timestamp,operation,operationMetrics
0,2026-11-20T21:56:01.244Z,CREATE TABLE AS SELECT,"Map(numFiles -> 1, numOutputRows -> 121317, numOutputBytes -> 3478254)"


In [0]:
product_path = (
    bronze_path + "production/product/Product.parquet"
)

product_bronze_df = spark.read.parquet(product_path)

product_bronze_df.printSchema()

print("Product Bronze row count:", product_bronze_df.count())

display(product_bronze_df.limit(10))

root
 |-- ProductID: integer (nullable = true)
 |-- Name: string (nullable = true)
 |-- ProductNumber: string (nullable = true)
 |-- MakeFlag: boolean (nullable = true)
 |-- FinishedGoodsFlag: boolean (nullable = true)
 |-- Color: string (nullable = true)
 |-- SafetyStockLevel: integer (nullable = true)
 |-- ReorderPoint: integer (nullable = true)
 |-- StandardCost: decimal(19,4) (nullable = true)
 |-- ListPrice: decimal(19,4) (nullable = true)
 |-- Size: string (nullable = true)
 |-- SizeUnitMeasureCode: string (nullable = true)
 |-- WeightUnitMeasureCode: string (nullable = true)
 |-- Weight: decimal(8,2) (nullable = true)
 |-- DaysToManufacture: integer (nullable = true)
 |-- ProductLine: string (nullable = true)
 |-- Class: string (nullable = true)
 |-- Style: string (nullable = true)
 |-- ProductSubcategoryID: integer (nullable = true)
 |-- ProductModelID: integer (nullable = true)
 |-- SellStartDate: timestamp (nullable = true)
 |-- SellEndDate: timestamp (nullable = true)
 |-- D

ProductID,Name,ProductNumber,MakeFlag,FinishedGoodsFlag,Color,SafetyStockLevel,ReorderPoint,StandardCost,ListPrice,Size,SizeUnitMeasureCode,WeightUnitMeasureCode,Weight,DaysToManufacture,ProductLine,Class,Style,ProductSubcategoryID,ProductModelID,SellStartDate,SellEndDate,DiscontinuedDate,rowguid,ModifiedDate
1,Adjustable Race,AR-5381,false,false,null,1000,750,0.0000,0.0000,null,null,null,null,0,null,null,null,null,null,2008-04-30T00:00:00Z,null,null,694215b7-08f7-4c0d-acb1-d734ba44c0c8,2014-02-08T10:01:36.827Z
2,Bearing Ball,BA-8327,false,false,null,1000,750,0.0000,0.0000,null,null,null,null,0,null,null,null,null,null,2008-04-30T00:00:00Z,null,null,58ae3c20-4f3a-4749-a7d4-d568806cc537,2014-02-08T10:01:36.827Z
3,BB Ball Bearing,BE-2349,true,false,null,800,600,0.0000,0.0000,null,null,null,null,1,null,null,null,null,null,2008-04-30T00:00:00Z,null,null,9c21aed2-5bfa-4f18-bcb8-f11638dc2e4e,2014-02-08T10:01:36.827Z
4,Headset Ball Bearings,BE-2908,false,false,null,800,600,0.0000,0.0000,null,null,null,null,0,null,null,null,null,null,2008-04-30T00:00:00Z,null,null,ecfed6cb-51ff-49b5-b06c-7d8ac834db8b,2014-02-08T10:01:36.827Z
316,Blade,BL-2036,true,false,null,800,600,0.0000,0.0000,null,null,null,null,1,null,null,null,null,null,2008-04-30T00:00:00Z,null,null,e73e9750-603b-4131-89f5-3dd15ed5ff80,2014-02-08T10:01:36.827Z
317,LL Crankarm,CA-5965,false,false,Black,500,375,0.0000,0.0000,null,null,null,null,0,null,L,null,null,null,2008-04-30T00:00:00Z,null,null,3c9d10b7-a6b2-4774-9963-c19dcee72fea,2014-02-08T10:01:36.827Z
318,ML Crankarm,CA-6738,false,false,Black,500,375,0.0000,0.0000,null,null,null,null,0,null,M,null,null,null,2008-04-30T00:00:00Z,null,null,eabb9a92-fa07-4eab-8955-f0517b4a4ca7,2014-02-08T10:01:36.827Z
319,HL Crankarm,CA-7457,false,false,Black,500,375,0.0000,0.0000,null,null,null,null,0,null,null,null,null,null,2008-04-30T00:00:00Z,null,null,7d3fd384-4f29-484b-86fa-4206e276fe58,2014-02-08T10:01:36.827Z
320,Chainring Bolts,CB-2903,false,false,Silver,1000,750,0.0000,0.0000,null,null,null,null,0,null,null,null,null,null,2008-04-30T00:00:00Z,null,null,7be38e48-b7d6-4486-888e-f53c26735101,2014-02-08T10:01:36.827Z
321,Chainring Nut,CN-6137,false,false,Silver,1000,750,0.0000,0.0000,null,null,null,null,0,null,null,null,null,null,2008-04-30T00:00:00Z,null,null,3314b1d7-ef69-4431-b6dd-dc75268bd5df,2014-02-08T10:01:36.827Z


In [0]:
from pyspark.sql import functions as F

product_quality_check = product_bronze_df.agg(
    F.count("*").alias("TotalRows"),

    F.countDistinct("ProductID").alias("DistinctProductIDs"),

    F.sum(
        F.when(F.col("ProductID").isNull(), 1).otherwise(0)
    ).alias("NullProductIDs"),

    F.sum(
        F.when(F.col("Name").isNull(), 1).otherwise(0)
    ).alias("NullNames"),

    F.sum(
        F.when(F.col("ProductNumber").isNull(), 1).otherwise(0)
    ).alias("NullProductNumbers"),

    F.sum(
        F.when(F.col("StandardCost") < 0, 1).otherwise(0)
    ).alias("NegativeStandardCosts"),

    F.sum(
        F.when(F.col("ListPrice") < 0, 1).otherwise(0)
    ).alias("NegativeListPrices"),

    F.sum(
        F.when(F.col("SafetyStockLevel") < 0, 1).otherwise(0)
    ).alias("NegativeSafetyStock"),

    F.sum(
        F.when(F.col("ReorderPoint") < 0, 1).otherwise(0)
    ).alias("NegativeReorderPoints"),

    F.sum(
        F.when(F.col("DaysToManufacture") < 0, 1).otherwise(0)
    ).alias("NegativeManufacturingDays")
)

display(product_quality_check)

TotalRows,DistinctProductIDs,NullProductIDs,NullNames,NullProductNumbers,NegativeStandardCosts,NegativeListPrices,NegativeSafetyStock,NegativeReorderPoints,NegativeManufacturingDays
504,504,0,0,0,0,0,0,0,0


In [0]:
product_key_check = product_bronze_df.agg(
    F.countDistinct("ProductNumber").alias("DistinctProductNumbers"),
    F.sum(
        F.when(
            F.col("SellEndDate").isNotNull()
            & (F.col("SellEndDate") < F.col("SellStartDate")),
            1
        ).otherwise(0)
    ).alias("SellEndBeforeSellStart")
)

display(product_key_check)

DistinctProductNumbers,SellEndBeforeSellStart
504,0


In [0]:
silver_salesorderdetail = spark.table(
    "adb_adventureworks_dev.silver.salesorderdetail"
)

unmatched_products = silver_salesorderdetail.join(
    product_bronze_df.select("ProductID"),
    on="ProductID",
    how="left_anti"
)

print("Unmatched Product references:", unmatched_products.count())

Unmatched Product references: 0


In [0]:
product_bronze_df.createOrReplaceTempView("product_silver_stage")

spark.sql("""
CREATE TABLE adb_adventureworks_dev.silver.product
USING DELTA
AS
SELECT *
FROM product_silver_stage
""")

DataFrame[num_affected_rows: bigint, num_inserted_rows: bigint]

In [0]:
silver_product = spark.table(
    "adb_adventureworks_dev.silver.product"
)

silver_product_check = silver_product.agg(
    F.count("*").alias("TotalRows"),
    F.countDistinct("ProductID").alias("DistinctProductIDs"),
    F.countDistinct("ProductNumber").alias("DistinctProductNumbers"),
    F.sum(F.when(F.col("ProductID").isNull(), 1).otherwise(0)).alias("NullProductIDs"),
    F.sum(F.when(F.col("ProductNumber").isNull(), 1).otherwise(0)).alias("NullProductNumbers")
)

display(silver_product_check)

TotalRows,DistinctProductIDs,DistinctProductNumbers,NullProductIDs,NullProductNumbers
504,504,504,0,0


In [0]:
unmatched_silver_products = (
    spark.table("adb_adventureworks_dev.silver.salesorderdetail")
    .join(
        silver_product.select("ProductID"),
        on="ProductID",
        how="left_anti"
    )
)

print(
    "Unmatched Silver Product references:",
    unmatched_silver_products.count()
)

Unmatched Silver Product references: 0


In [0]:
product_incremental_path = (
    bronze_path + "production/product/incremental/"
)

product_incremental_df = spark.read.parquet(
    product_incremental_path
)

print("Incremental rows:", product_incremental_df.count())

display(product_incremental_df)

Incremental rows: 1


ProductID,Name,ProductNumber,MakeFlag,FinishedGoodsFlag,Color,SafetyStockLevel,ReorderPoint,StandardCost,ListPrice,Size,SizeUnitMeasureCode,WeightUnitMeasureCode,Weight,DaysToManufacture,ProductLine,Class,Style,ProductSubcategoryID,ProductModelID,SellStartDate,SellEndDate,DiscontinuedDate,rowguid,ModifiedDate
940,HL Road Pedal,PD-R853,false,true,Silver/Black,500,375,35.9596,80.9900,null,null,G,149.00,1,R,H,null,13,70,2013-05-30T00:00:00Z,null,null,44e96967-ab99-41ed-8b41-5bc70a5ca1a9,2014-02-08T10:03:55.51Z


In [0]:
product_incremental_new_records = (
    product_incremental_df.exceptAll(silver_product)
)

product_incremental_new_records_count = (
    product_incremental_new_records.count()
)

print(
    "Incremental records not already present in Silver:",
    product_incremental_new_records_count
)

display(product_incremental_new_records)

Incremental records not already present in Silver: 0


ProductID,Name,ProductNumber,MakeFlag,FinishedGoodsFlag,Color,SafetyStockLevel,ReorderPoint,StandardCost,ListPrice,Size,SizeUnitMeasureCode,WeightUnitMeasureCode,Weight,DaysToManufacture,ProductLine,Class,Style,ProductSubcategoryID,ProductModelID,SellStartDate,SellEndDate,DiscontinuedDate,rowguid,ModifiedDate


In [0]:
display(
    spark.sql("""
        DESCRIBE DETAIL adb_adventureworks_dev.silver.product
    """)
)

display(
    spark.sql("""
        DESCRIBE HISTORY adb_adventureworks_dev.silver.product
    """)
)

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,3e43a52b-0c63-4ae6-8857-d1c51cd61378,adb_adventureworks_dev.silver.product,null,abfss://unity-catalog-storage@dbstorage7xahujkjryzy4.dfs.core.windows.net/7405604737487716/__unitystorage/catalogs/5d8d9234-455a-4e08-9d0b-e9030e8d5e6d/tables/46edca6a-fdf6-4c2d-9c52-5c03a3706ee0,2026-11-20T22:43:21.699Z,2026-11-20T22:43:27Z,List(),List(),1,25974,"Map(delta.parquet.compression.codec -> zstd, delta.enableDeletionVectors -> true, delta.parquet.format.version -> 2.12.0, delta.enableRowTracking -> true, delta.checkpointPolicy -> v2, io.unitycatalog.tableId -> 46edca6a-fdf6-4c2d-9c52-5c03a3706ee0, delta.rowTracking.materializedRowCommitVersionColumnName -> _row-commit-version-col-4bfa94c4-799f-4dd2-9c69-b3831542acb1, delta.rowTracking.materializedRowIdColumnName -> _row-id-col-1fd898f1-ca08-4d5f-a924-654cc4e5936e, delta.enableInCommitTimestamps -> true)",3,7,"List(appendOnly, clustering, deletionVectors, domainMetadata, inCommitTimestamp, invariants, rowTracking, v2Checkpoint)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",true


version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
0,2026-11-20T22:43:27.429Z,142540505220658,lehlogonoloramphisa@outlook.com,CREATE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""databricks.internal.autoUpgrades.delta.enableRowTracking"":""true"",""delta.enableDeletionVectors"":""true"",""databricks.internal.autoUpgrades.delta.checkpointPolicy"":""v2"",""databricks.internal.autoUpgrades.delta.parquet.format.version"":""2.12.0"",""delta.parquet.format.version"":""2.12.0"",""delta.enableRowTracking"":""true"",""delta.checkpointPolicy"":""v2"",""io.unitycatalog.tableId"":""46edca6a-fdf6-4c2d-9c52-5c03a3706ee0"",""delta.rowTracking.materializedRowCommitVersionColumnName"":""_row-commit-version-col-4bfa94c4-799f-4dd2-9c69-b3831542acb1"",""delta.enableInCommitTimestamps"":""true"",""databricks.internal.autoUpgrades.delta.enableInCommitTimestamps"":""true"",""delta.rowTracking.materializedRowIdColumnName"":""_row-id-col-1fd898f1-ca08-4d5f-a924-654cc4e5936e""}, statsOnLoad -> true, clusterByAuto -> true, clusteringOnWriteStatus -> Reason for skipping: Nonexistent clustering column, generally when replacing a Liquid table with a table without clustering columns)",null,List(1361742418849894),null,1120-175118-j84pd9yh,null,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 504, numOutputBytes -> 25974)",null,Databricks-Runtime/18.x-photon-scala2.13


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

product_merge_source = product_incremental_df

duplicate_product_ids = (
    product_merge_source
    .groupBy("ProductID")
    .count()
    .filter(F.col("count") > 1)
)

print("Duplicate ProductIDs:", duplicate_product_ids.count())
print("Source rows:", product_merge_source.count())

Duplicate ProductIDs: 0
Source rows: 1


In [0]:
product_merge_comparison = (
    product_merge_source.alias("source")
    .join(
        silver_product.alias("target"),
        on="ProductID",
        how="inner"
    )
    .select(
        F.col("source.ProductID"),
        F.col("source.Name").alias("SourceName"),
        F.col("target.Name").alias("TargetName"),
        F.col("source.ModifiedDate").alias("SourceModifiedDate"),
        F.col("target.ModifiedDate").alias("TargetModifiedDate")
    )
)

display(product_merge_comparison)

ProductID,SourceName,TargetName,SourceModifiedDate,TargetModifiedDate
940,HL Road Pedal,HL Road Pedal,2014-02-08T10:03:55.51Z,2014-02-08T10:03:55.51Z


In [0]:
from delta.tables import DeltaTable

product_delta_table = DeltaTable.forName(
    spark,
    "adb_adventureworks_dev.silver.product"
)

(
    product_delta_table.alias("target")
    .merge(
        product_merge_source.alias("source"),
        "target.ProductID = source.ProductID"
    )
    .whenMatchedUpdateAll(
        condition="source.ModifiedDate > target.ModifiedDate"
    )
    .whenNotMatchedInsertAll()
    .execute()
)

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
display(
    spark.sql("""
        DESCRIBE HISTORY adb_adventureworks_dev.silver.product
        LIMIT 1
    """).select(
        "version",
        "operation",
        "operationMetrics"
    )
)

version,operation,operationMetrics
1,MERGE,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 0, numTargetBytesAdded -> 0, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 0, executionTimeMs -> 11853, materializeSourceTimeMs -> 659, numTargetRowsInserted -> 0, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 4434, numTargetRowsUpdated -> 0, numOutputRows -> 0, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 1, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 6531)"


In [0]:
spark.sql("""
CREATE TABLE adb_adventureworks_dev.silver.product_merge_test
USING DELTA
AS
SELECT *
FROM adb_adventureworks_dev.silver.product
""")

DataFrame[num_affected_rows: bigint, num_inserted_rows: bigint]

In [0]:
product_merge_test = spark.table(
    "adb_adventureworks_dev.silver.product_merge_test"
)

print("Test table rows:", product_merge_test.count())
print(
    "Distinct ProductIDs:",
    product_merge_test.select("ProductID").distinct().count()
)

Test table rows: 504
Distinct ProductIDs: 504


In [0]:
from datetime import datetime
from decimal import Decimal

product_merge_test_source = spark.createDataFrame(
    [
        (
            940,
            "HL Road Pedal Updated",
            "PD-R853",
            False,
            True,
            "Silver/Black",
            500,
            375,
            Decimal("35.9596"),
            Decimal("80.9900"),
            None,
            None,
            "G",
            Decimal("149.00"),
            1,
            "R",
            "H",
            None,
            13,
            70,
            datetime(2013, 5, 30),
            None,
            None,
            "44e96967-ab99-41ed-8b41-5bc70a5ca1a9",
            datetime(2015, 1, 1)
        ),
        (
            99999,
            "Test Product",
            "TEST-99999",
            False,
            False,
            "Blue",
            100,
            50,
            Decimal("10.0000"),
            Decimal("20.0000"),
            None,
            None,
            None,
            None,
            0,
            None,
            None,
            None,
            None,
            None,
            datetime(2026, 1, 1),
            None,
            None,
            "00000000-0000-0000-0000-000000000001",
            datetime(2026, 1, 1)
        )
    ],
    product_bronze_df.schema
)

display(product_merge_test_source)

ProductID,Name,ProductNumber,MakeFlag,FinishedGoodsFlag,Color,SafetyStockLevel,ReorderPoint,StandardCost,ListPrice,Size,SizeUnitMeasureCode,WeightUnitMeasureCode,Weight,DaysToManufacture,ProductLine,Class,Style,ProductSubcategoryID,ProductModelID,SellStartDate,SellEndDate,DiscontinuedDate,rowguid,ModifiedDate
940,HL Road Pedal Updated,PD-R853,false,true,Silver/Black,500,375,35.9596,80.9900,null,null,G,149.00,1,R,H,null,13,70,2013-05-30T00:00:00Z,null,null,44e96967-ab99-41ed-8b41-5bc70a5ca1a9,2015-01-01T00:00:00Z
99999,Test Product,TEST-99999,false,false,Blue,100,50,10.0000,20.0000,null,null,null,null,0,null,null,null,null,null,2026-01-01T00:00:00Z,null,null,00000000-0000-0000-0000-000000000001,2026-01-01T00:00:00Z


In [0]:
product_test_delta = DeltaTable.forName(
    spark,
    "adb_adventureworks_dev.silver.product_merge_test"
)

(
    product_test_delta.alias("target")
    .merge(
        product_merge_test_source.alias("source"),
        "target.ProductID = source.ProductID"
    )
    .whenMatchedUpdateAll(
        condition="source.ModifiedDate > target.ModifiedDate"
    )
    .whenNotMatchedInsertAll()
    .execute()
)

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
display(
    spark.sql("""
        DESCRIBE HISTORY adb_adventureworks_dev.silver.product_merge_test
        LIMIT 1
    """).select(
        "version",
        "operation",
        "operationMetrics"
    )
)


version,operation,operationMetrics
1,MERGE,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 2, numTargetBytesAdded -> 13663, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 1, executionTimeMs -> 6419, materializeSourceTimeMs -> 218, numTargetRowsInserted -> 1, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 2315, numTargetRowsUpdated -> 1, numOutputRows -> 2, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 2, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 3844)"


In [0]:
display(
    spark.table(
        "adb_adventureworks_dev.silver.product_merge_test"
    )
    .filter(F.col("ProductID").isin(940, 99999))
    .select(
        "ProductID",
        "Name",
        "ProductNumber",
        "ModifiedDate"
    )
    .orderBy("ProductID")
)

ProductID,Name,ProductNumber,ModifiedDate
940,HL Road Pedal Updated,PD-R853,2015-01-01T00:00:00Z
99999,Test Product,TEST-99999,2026-01-01T00:00:00Z


In [0]:
product_merge_test_final = spark.table(
    "adb_adventureworks_dev.silver.product_merge_test"
)

print("Total rows:", product_merge_test_final.count())

print(
    "Distinct ProductIDs:",
    product_merge_test_final.select("ProductID").distinct().count()
)

Total rows: 505
Distinct ProductIDs: 505


In [0]:
%sql
DROP TABLE adb_adventureworks_dev.silver.product_merge_test;

In [0]:
customer_incremental_path = (
    bronze_path + "sales/customer/incremental/"
)

customer_incremental_df = (
    spark.read
    .format("parquet")
    .load(customer_incremental_path)
)

print("Total incremental rows:", customer_incremental_df.count())

print(
    "Distinct CustomerIDs:",
    customer_incremental_df.select("CustomerID").distinct().count()
)

display(
    customer_incremental_df
    .select("CustomerID", "ModifiedDate")
    .orderBy("CustomerID", "ModifiedDate")
)

Total incremental rows: 79290
Distinct CustomerIDs: 19820


CustomerID,ModifiedDate
1,2014-09-12T11:15:07.263Z
1,2014-09-12T11:15:07.263Z
1,2014-09-12T11:15:07.263Z
1,2014-09-13T00:00:00Z
1,2014-09-13T00:00:00Z
2,2014-09-12T11:15:07.263Z
2,2014-09-12T11:15:07.263Z
2,2014-09-12T11:15:07.263Z
2,2014-09-12T11:15:07.263Z
2,2014-09-14T10:00:00Z


In [0]:
from pyspark.sql import functions as F

customer_file_counts = (
    customer_incremental_df
    .select(
        F.col("_metadata.file_path").alias("SourceFile")
    )
    .groupBy("SourceFile")
    .count()
    .orderBy("SourceFile")
)

display(customer_file_counts)

SourceFile,count
abfss://bronze@awdata2022.dfs.core.windows.net/sales/customer/incremental/Customer_2b1d3b82-c0d2-4433-8466-42d121c4c664.parquet,4
abfss://bronze@awdata2022.dfs.core.windows.net/sales/customer/incremental/Customer_35f97c3e-cffc-40d0-862e-b0685d6c35e1.parquet,3
abfss://bronze@awdata2022.dfs.core.windows.net/sales/customer/incremental/Customer_49a11346-0a72-4916-9e10-41b3ceaded18.parquet,19820
abfss://bronze@awdata2022.dfs.core.windows.net/sales/customer/incremental/Customer_6cac908b-e9ba-422b-b243-00e7fc7bd56e.parquet,3
abfss://bronze@awdata2022.dfs.core.windows.net/sales/customer/incremental/Customer_77efa261-e26e-4ce8-acb0-034232871915.parquet,19820
abfss://bronze@awdata2022.dfs.core.windows.net/sales/customer/incremental/Customer_7d981eec-44ea-4bf7-97c3-f76819c11244.parquet,19820
abfss://bronze@awdata2022.dfs.core.windows.net/sales/customer/incremental/Customer_c931e768-2d15-43b3-8ccd-c7c7902c5d45.parquet,19820


In [0]:
small_customer_files = [
    "Customer_2b1d3b82-c0d2-4433-8466-42d121c4c664.parquet",
    "Customer_35f97c3e-cffc-40d0-862e-b0685d6c35e1.parquet",
    "Customer_6cac908b-e9ba-422b-b243-00e7fc7bd56e.parquet"
]

small_customer_paths = [
    bronze_path + "sales/customer/incremental/" + file
    for file in small_customer_files
]

small_customer_df = (
    spark.read
    .format("parquet")
    .load(small_customer_paths)
)

display(
    small_customer_df
    .select("CustomerID", "AccountNumber", "ModifiedDate")
    .orderBy("CustomerID", "ModifiedDate")
)

CustomerID,AccountNumber,ModifiedDate
1,AW00000001,2014-09-13T00:00:00Z
2,AW00000002,2014-09-14T10:00:00Z
2,AW00000002,2014-09-14T10:00:00Z
2,AW00000002,2014-09-14T10:00:00Z
3,AW00000003,2014-09-14T10:00:00Z
3,AW00000003,2014-09-14T10:00:00Z
3,AW00000003,2014-09-14T10:00:00Z
4,AW00000004,2014-09-14T10:00:00Z
4,AW00000004,2014-09-14T10:00:00Z
4,AW00000004,2014-09-14T10:00:00Z


In [0]:
from pyspark.sql import functions as F

customer_columns = small_customer_df.columns

customer_version_check = (
    small_customer_df
    .groupBy("CustomerID", "ModifiedDate")
    .agg(
        F.count("*").alias("ExtractedCopies"),
        F.countDistinct(
            F.struct(*[F.col(c) for c in customer_columns])
        ).alias("DistinctFullRecords")
    )
    .orderBy("CustomerID", "ModifiedDate")
)

display(customer_version_check)

CustomerID,ModifiedDate,ExtractedCopies,DistinctFullRecords
1,2014-09-13T00:00:00Z,1,1
2,2014-09-14T10:00:00Z,3,1
3,2014-09-14T10:00:00Z,3,1
4,2014-09-14T10:00:00Z,3,1


In [0]:
customer_merge_source = (
    small_customer_df
    .dropDuplicates(["CustomerID", "ModifiedDate"])
)

print("Rows before deduplication:", small_customer_df.count())
print("Rows after deduplication:", customer_merge_source.count())
print(
    "Distinct CustomerIDs:",
    customer_merge_source.select("CustomerID").distinct().count()
)

display(
    customer_merge_source
    .select("CustomerID", "AccountNumber", "ModifiedDate")
    .orderBy("CustomerID")
)

Rows before deduplication: 10
Rows after deduplication: 4
Distinct CustomerIDs: 4


CustomerID,AccountNumber,ModifiedDate
1,AW00000001,2014-09-13T00:00:00Z
2,AW00000002,2014-09-14T10:00:00Z
3,AW00000003,2014-09-14T10:00:00Z
4,AW00000004,2014-09-14T10:00:00Z


In [0]:
from pyspark.sql import functions as F

customer_silver_df = spark.table(
    "adb_adventureworks_dev.silver.customer"
)

customer_comparison = (
    customer_merge_source.alias("source")
    .join(
        customer_silver_df.alias("target"),
        on="CustomerID",
        how="left"
    )
    .select(
        F.col("source.CustomerID"),
        F.col("source.ModifiedDate").alias("SourceModifiedDate"),
        F.col("target.ModifiedDate").alias("SilverModifiedDate"),
        F.when(
            F.col("target.CustomerID").isNull(),
            "INSERT"
        ).when(
            F.col("source.ModifiedDate") >
            F.col("target.ModifiedDate"),
            "UPDATE"
        ).when(
            F.col("source.ModifiedDate") ==
            F.col("target.ModifiedDate"),
            "NO CHANGE"
        ).otherwise("OLDER SOURCE")
        .alias("MergeAction")
    )
    .orderBy("CustomerID")
)

display(customer_comparison)

CustomerID,SourceModifiedDate,SilverModifiedDate,MergeAction
1,2014-09-13T00:00:00Z,2014-09-13T00:00:00Z,NO CHANGE
2,2014-09-14T10:00:00Z,2014-09-14T10:00:00Z,NO CHANGE
3,2014-09-14T10:00:00Z,2014-09-14T10:00:00Z,NO CHANGE
4,2014-09-14T10:00:00Z,2014-09-14T10:00:00Z,NO CHANGE


In [0]:
from delta.tables import DeltaTable

customer_delta_table = DeltaTable.forName(
    spark,
    "adb_adventureworks_dev.silver.customer"
)

(
    customer_delta_table.alias("target")
    .merge(
        customer_merge_source.alias("source"),
        "target.CustomerID = source.CustomerID"
    )
    .whenMatchedUpdateAll(
        condition="source.ModifiedDate > target.ModifiedDate"
    )
    .whenNotMatchedInsertAll()
    .execute()
)

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
customer_history = DeltaTable.forName(
    spark,
    "adb_adventureworks_dev.silver.customer"
).history(1)

display(
    customer_history.select(
        "version",
        "operation",
        "operationMetrics"
    )
)

version,operation,operationMetrics
1,MERGE,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 0, numTargetBytesAdded -> 0, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 0, executionTimeMs -> 4155, materializeSourceTimeMs -> 558, numTargetRowsInserted -> 0, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 2115, numTargetRowsUpdated -> 0, numOutputRows -> 0, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 4, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 1436)"


In [0]:
customer_final_df = spark.table(
    "adb_adventureworks_dev.silver.customer"
)

print("Total rows:", customer_final_df.count())
print(
    "Distinct CustomerIDs:",
    customer_final_df.select("CustomerID").distinct().count()
)
print(
    "Duplicate CustomerIDs:",
    customer_final_df.count()
    - customer_final_df.select("CustomerID").distinct().count()
)

Total rows: 19820
Distinct CustomerIDs: 19820
Duplicate CustomerIDs: 0


In [0]:
from pyspark.sql import functions as F

header_incremental_path = (
    bronze_path + "sales/salesorderheader/incremental/"
)

header_incremental_df = (
    spark.read
    .format("parquet")
    .load(header_incremental_path)
)

header_file_counts = (
    header_incremental_df
    .groupBy(F.col("_metadata.file_path").alias("FilePath"))
    .count()
    .orderBy("FilePath")
)

display(header_file_counts)

print("Total incremental rows:", header_incremental_df.count())

print(
    "Distinct SalesOrderIDs:",
    header_incremental_df
    .select("SalesOrderID")
    .distinct()
    .count()
)

FilePath,count
abfss://bronze@awdata2022.dfs.core.windows.net/sales/salesorderheader/incremental/SalesOrderHeader_2b1d3b82-c0d2-4433-8466-42d121c4c664.parquet,40
abfss://bronze@awdata2022.dfs.core.windows.net/sales/salesorderheader/incremental/SalesOrderHeader_35f97c3e-cffc-40d0-862e-b0685d6c35e1.parquet,40
abfss://bronze@awdata2022.dfs.core.windows.net/sales/salesorderheader/incremental/SalesOrderHeader_4d2e4808-6497-4eb1-8a6c-59088fe08359.parquet,40
abfss://bronze@awdata2022.dfs.core.windows.net/sales/salesorderheader/incremental/SalesOrderHeader_6cac908b-e9ba-422b-b243-00e7fc7bd56e.parquet,40
abfss://bronze@awdata2022.dfs.core.windows.net/sales/salesorderheader/incremental/SalesOrderHeader_77efa261-e26e-4ce8-acb0-034232871915.parquet,40
abfss://bronze@awdata2022.dfs.core.windows.net/sales/salesorderheader/incremental/SalesOrderHeader_7d981eec-44ea-4bf7-97c3-f76819c11244.parquet,40
abfss://bronze@awdata2022.dfs.core.windows.net/sales/salesorderheader/incremental/SalesOrderHeader_c931e768-2d15-43b3-8ccd-c7c7902c5d45.parquet,40


Total incremental rows: 280
Distinct SalesOrderIDs: 40


In [0]:
header_version_check = (
    header_incremental_df
    .groupBy("SalesOrderID", "ModifiedDate")
    .agg(
        F.count("*").alias("ExtractedCopies"),
        F.countDistinct(
            F.struct(*[
                F.col(c)
                for c in header_incremental_df.columns
                if c != "_metadata"
            ])
        ).alias("DistinctFullRecords")
    )
    .orderBy("SalesOrderID", "ModifiedDate")
)

display(header_version_check)

print(
    "Unique SalesOrderID and ModifiedDate combinations:",
    header_version_check.count()
)

print(
    "Conflicting versions:",
    header_version_check
    .filter(F.col("DistinctFullRecords") > 1)
    .count()
)

SalesOrderID,ModifiedDate,ExtractedCopies,DistinctFullRecords
75084,2014-07-07T00:00:00Z,7,1
75085,2014-07-07T00:00:00Z,7,1
75086,2014-07-07T00:00:00Z,7,1
75087,2014-07-07T00:00:00Z,7,1
75088,2014-07-07T00:00:00Z,7,1
75089,2014-07-07T00:00:00Z,7,1
75090,2014-07-07T00:00:00Z,7,1
75091,2014-07-07T00:00:00Z,7,1
75092,2014-07-07T00:00:00Z,7,1
75093,2014-07-07T00:00:00Z,7,1


Unique SalesOrderID and ModifiedDate combinations: 40
Conflicting versions: 0


In [0]:
header_merge_source = (
    header_incremental_df
    .dropDuplicates(["SalesOrderID", "ModifiedDate"])
)

print("Rows before deduplication:", header_incremental_df.count())
print("Rows after deduplication:", header_merge_source.count())

print(
    "Distinct SalesOrderIDs:",
    header_merge_source.select("SalesOrderID").distinct().count()
)

display(
    header_merge_source
    .select("SalesOrderID", "OrderDate", "CustomerID", "ModifiedDate")
    .orderBy("SalesOrderID")
)

Rows before deduplication: 280
Rows after deduplication: 40
Distinct SalesOrderIDs: 40


SalesOrderID,OrderDate,CustomerID,ModifiedDate
75084,2014-06-30T00:00:00Z,11078,2014-07-07T00:00:00Z
75085,2014-06-30T00:00:00Z,11927,2014-07-07T00:00:00Z
75086,2014-06-30T00:00:00Z,28789,2014-07-07T00:00:00Z
75087,2014-06-30T00:00:00Z,11794,2014-07-07T00:00:00Z
75088,2014-06-30T00:00:00Z,14680,2014-07-07T00:00:00Z
75089,2014-06-30T00:00:00Z,19585,2014-07-07T00:00:00Z
75090,2014-06-30T00:00:00Z,27686,2014-07-07T00:00:00Z
75091,2014-06-30T00:00:00Z,20601,2014-07-07T00:00:00Z
75092,2014-06-30T00:00:00Z,26564,2014-07-07T00:00:00Z
75093,2014-06-30T00:00:00Z,16170,2014-07-07T00:00:00Z


In [0]:
header_silver_df = spark.table(
    "adb_adventureworks_dev.silver.salesorderheader"
)

header_comparison = (
    header_merge_source.alias("source")
    .join(
        header_silver_df.alias("target"),
        on="SalesOrderID",
        how="left"
    )
    .select(
        F.col("source.SalesOrderID"),
        F.col("source.ModifiedDate").alias("SourceModifiedDate"),
        F.col("target.ModifiedDate").alias("SilverModifiedDate"),
        F.when(
            F.col("target.SalesOrderID").isNull(),
            "INSERT"
        ).when(
            F.col("source.ModifiedDate") >
            F.col("target.ModifiedDate"),
            "UPDATE"
        ).when(
            F.col("source.ModifiedDate") ==
            F.col("target.ModifiedDate"),
            "NO CHANGE"
        ).otherwise("OLDER SOURCE")
        .alias("MergeAction")
    )
    .orderBy("SalesOrderID")
)

display(header_comparison)

print("Merge action counts:")
display(
    header_comparison
    .groupBy("MergeAction")
    .count()
    .orderBy("MergeAction")
)

SalesOrderID,SourceModifiedDate,SilverModifiedDate,MergeAction
75084,2014-07-07T00:00:00Z,2014-07-07T00:00:00Z,NO CHANGE
75085,2014-07-07T00:00:00Z,2014-07-07T00:00:00Z,NO CHANGE
75086,2014-07-07T00:00:00Z,2014-07-07T00:00:00Z,NO CHANGE
75087,2014-07-07T00:00:00Z,2014-07-07T00:00:00Z,NO CHANGE
75088,2014-07-07T00:00:00Z,2014-07-07T00:00:00Z,NO CHANGE
75089,2014-07-07T00:00:00Z,2014-07-07T00:00:00Z,NO CHANGE
75090,2014-07-07T00:00:00Z,2014-07-07T00:00:00Z,NO CHANGE
75091,2014-07-07T00:00:00Z,2014-07-07T00:00:00Z,NO CHANGE
75092,2014-07-07T00:00:00Z,2014-07-07T00:00:00Z,NO CHANGE
75093,2014-07-07T00:00:00Z,2014-07-07T00:00:00Z,NO CHANGE


Merge action counts:


MergeAction,count
NO CHANGE,40


In [0]:
header_delta_table = DeltaTable.forName(
    spark,
    "adb_adventureworks_dev.silver.salesorderheader"
)

(
    header_delta_table.alias("target")
    .merge(
        header_merge_source.alias("source"),
        "target.SalesOrderID = source.SalesOrderID"
    )
    .whenMatchedUpdateAll(
        condition="source.ModifiedDate > target.ModifiedDate"
    )
    .whenNotMatchedInsertAll()
    .execute()
)

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
header_history = DeltaTable.forName(
    spark,
    "adb_adventureworks_dev.silver.salesorderheader"
).history(1)

display(
    header_history.select(
        "version",
        "operation",
        "operationMetrics"
    )
)

version,operation,operationMetrics
1,MERGE,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 0, numTargetBytesAdded -> 0, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 0, executionTimeMs -> 5367, materializeSourceTimeMs -> 1630, numTargetRowsInserted -> 0, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 1864, numTargetRowsUpdated -> 0, numOutputRows -> 0, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 40, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 1833)"


In [0]:
header_final_df = spark.table(
    "adb_adventureworks_dev.silver.salesorderheader"
)

print("Total rows:", header_final_df.count())

print(
    "Distinct SalesOrderIDs:",
    header_final_df.select("SalesOrderID").distinct().count()
)

print(
    "Duplicate SalesOrderIDs:",
    header_final_df.count()
    - header_final_df.select("SalesOrderID").distinct().count()
)

Total rows: 31465
Distinct SalesOrderIDs: 31465
Duplicate SalesOrderIDs: 0


In [0]:
from pyspark.sql import functions as F

detail_incremental_path = (
    "abfss://bronze@awdata2022.dfs.core.windows.net/"
    "sales/salesorderdetail/incremental/"
)

detail_incremental_df = (
    spark.read
    .parquet(detail_incremental_path)
    .withColumn(
        "SourceFile",
        F.col("_metadata.file_path")
    )
)

display(
    detail_incremental_df
    .groupBy("SourceFile")
    .agg(
        F.count("*").alias("RowCount"),
        F.countDistinct("SalesOrderID").alias("DistinctSalesOrderIDs"),
        F.countDistinct("SalesOrderDetailID").alias("DistinctDetailIDs"),
        F.min("ModifiedDate").alias("MinModifiedDate"),
        F.max("ModifiedDate").alias("MaxModifiedDate")
    )
    .orderBy("SourceFile")
)

SourceFile,RowCount,DistinctSalesOrderIDs,DistinctDetailIDs,MinModifiedDate,MaxModifiedDate
abfss://bronze@awdata2022.dfs.core.windows.net/sales/salesorderdetail/incremental/SalesOrderDetail_2b1d3b82-c0d2-4433-8466-42d121c4c664.parquet,96,40,96,2014-06-30T00:00:00Z,2014-06-30T00:00:00Z
abfss://bronze@awdata2022.dfs.core.windows.net/sales/salesorderdetail/incremental/SalesOrderDetail_35f97c3e-cffc-40d0-862e-b0685d6c35e1.parquet,96,40,96,2014-06-30T00:00:00Z,2014-06-30T00:00:00Z
abfss://bronze@awdata2022.dfs.core.windows.net/sales/salesorderdetail/incremental/SalesOrderDetail_4d2e4808-6497-4eb1-8a6c-59088fe08359.parquet,96,40,96,2014-06-30T00:00:00Z,2014-06-30T00:00:00Z
abfss://bronze@awdata2022.dfs.core.windows.net/sales/salesorderdetail/incremental/SalesOrderDetail_6cac908b-e9ba-422b-b243-00e7fc7bd56e.parquet,96,40,96,2014-06-30T00:00:00Z,2014-06-30T00:00:00Z
abfss://bronze@awdata2022.dfs.core.windows.net/sales/salesorderdetail/incremental/SalesOrderDetail_77efa261-e26e-4ce8-acb0-034232871915.parquet,96,40,96,2014-06-30T00:00:00Z,2014-06-30T00:00:00Z
abfss://bronze@awdata2022.dfs.core.windows.net/sales/salesorderdetail/incremental/SalesOrderDetail_7d981eec-44ea-4bf7-97c3-f76819c11244.parquet,96,40,96,2014-06-30T00:00:00Z,2014-06-30T00:00:00Z
abfss://bronze@awdata2022.dfs.core.windows.net/sales/salesorderdetail/incremental/SalesOrderDetail_c931e768-2d15-43b3-8ccd-c7c7902c5d45.parquet,96,40,96,2014-06-30T00:00:00Z,2014-06-30T00:00:00Z


In [0]:
detail_version_check = (
    detail_incremental_df
    .groupBy(
        "SalesOrderID",
        "SalesOrderDetailID",
        "ModifiedDate"
    )
    .agg(
        F.count("*").alias("OccurrenceCount"),
        F.countDistinct(
            F.to_json(
                F.struct(*[
                    F.col(c)
                    for c in detail_incremental_df.columns
                    if c != "SourceFile"
                ])
            )
        ).alias("DistinctFullRecords")
    )
)

display(
    detail_version_check
    .groupBy("DistinctFullRecords")
    .agg(
        F.count("*").alias("CompositeKeyVersions")
    )
)

DistinctFullRecords,CompositeKeyVersions
1,96


In [0]:
detail_merge_source = (
    detail_incremental_df
    .dropDuplicates([
        "SalesOrderID",
        "SalesOrderDetailID",
        "ModifiedDate"
    ])
    .drop("SourceFile")
)

print("Rows before deduplication:", detail_incremental_df.count())
print("Rows after deduplication:", detail_merge_source.count())

print(
    "Distinct composite keys:",
    detail_merge_source
    .select("SalesOrderID", "SalesOrderDetailID")
    .distinct()
    .count()
)

Rows before deduplication: 672
Rows after deduplication: 96
Distinct composite keys: 96


In [0]:
detail_silver_df = spark.table(
    "adb_adventureworks_dev.silver.salesorderdetail"
)

detail_comparison = (
    detail_merge_source.alias("source")
    .join(
        detail_silver_df.alias("target"),
        on=["SalesOrderID", "SalesOrderDetailID"],
        how="left"
    )
    .select(
        F.col("source.SalesOrderID"),
        F.col("source.SalesOrderDetailID"),
        F.col("source.ModifiedDate").alias("SourceModifiedDate"),
        F.col("target.ModifiedDate").alias("SilverModifiedDate"),
        F.when(
            F.col("target.SalesOrderID").isNull(),
            "INSERT"
        )
        .when(
            F.col("source.ModifiedDate") > F.col("target.ModifiedDate"),
            "UPDATE"
        )
        .when(
            F.col("source.ModifiedDate") == F.col("target.ModifiedDate"),
            "NO CHANGE"
        )
        .otherwise("OLDER SOURCE")
        .alias("MergeAction")
    )
)

display(
    detail_comparison
    .groupBy("MergeAction")
    .count()
)

MergeAction,count
NO CHANGE,96


In [0]:
from delta.tables import DeltaTable

detail_delta_table = DeltaTable.forName(
    spark,
    "adb_adventureworks_dev.silver.salesorderdetail"
)

(
    detail_delta_table.alias("target")
    .merge(
        detail_merge_source.alias("source"),
        """
        target.SalesOrderID = source.SalesOrderID
        AND target.SalesOrderDetailID = source.SalesOrderDetailID
        """
    )
    .whenMatchedUpdateAll(
        condition="source.ModifiedDate > target.ModifiedDate"
    )
    .whenNotMatchedInsertAll()
    .execute()
)

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
detail_history = DeltaTable.forName(
    spark,
    "adb_adventureworks_dev.silver.salesorderdetail"
).history(1)

display(
    detail_history.select(
        "version",
        "operation",
        "operationMetrics"
    )
)

version,operation,operationMetrics
1,MERGE,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 0, numTargetBytesAdded -> 0, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 0, executionTimeMs -> 4964, materializeSourceTimeMs -> 530, numTargetRowsInserted -> 0, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 2918, numTargetRowsUpdated -> 0, numOutputRows -> 0, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 96, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 1480)"


In [0]:
detail_final_df = spark.table(
    "adb_adventureworks_dev.silver.salesorderdetail"
)

total_rows = detail_final_df.count()

distinct_keys = (
    detail_final_df
    .select("SalesOrderID", "SalesOrderDetailID")
    .distinct()
    .count()
)

print("Total rows:", total_rows)
print("Distinct composite keys:", distinct_keys)
print("Duplicate composite keys:", total_rows - distinct_keys)

Total rows: 121317
Distinct composite keys: 121317
Duplicate composite keys: 0


In [0]:
silver_tables = [
    "customer",
    "salesorderheader",
    "salesorderdetail",
    "product"
]

for table_name in silver_tables:
    print(f"\n{'=' * 60}")
    print(f"TABLE: {table_name}")
    print(f"{'=' * 60}")

    spark.table(
        f"adb_adventureworks_dev.silver.{table_name}"
    ).printSchema()


TABLE: customer
root
 |-- CustomerID: integer (nullable = true)
 |-- PersonID: integer (nullable = true)
 |-- StoreID: integer (nullable = true)
 |-- TerritoryID: integer (nullable = true)
 |-- AccountNumber: string (nullable = true)
 |-- rowguid: string (nullable = true)
 |-- ModifiedDate: timestamp (nullable = true)


TABLE: salesorderheader
root
 |-- SalesOrderID: integer (nullable = true)
 |-- RevisionNumber: integer (nullable = true)
 |-- OrderDate: timestamp (nullable = true)
 |-- DueDate: timestamp (nullable = true)
 |-- ShipDate: timestamp (nullable = true)
 |-- Status: integer (nullable = true)
 |-- OnlineOrderFlag: boolean (nullable = true)
 |-- SalesOrderNumber: string (nullable = true)
 |-- PurchaseOrderNumber: string (nullable = true)
 |-- AccountNumber: string (nullable = true)
 |-- CustomerID: integer (nullable = true)
 |-- SalesPersonID: integer (nullable = true)
 |-- TerritoryID: integer (nullable = true)
 |-- BillToAddressID: integer (nullable = true)
 |-- ShipToAddr

In [0]:
from pyspark.sql import functions as F

customers = spark.table("adb_adventureworks_dev.silver.customer")
headers = spark.table("adb_adventureworks_dev.silver.salesorderheader")
details = spark.table("adb_adventureworks_dev.silver.salesorderdetail")
products = spark.table("adb_adventureworks_dev.silver.product")

detail_count = details.count()

checks = [
    (
        "Sales details without matching order",
        details.join(
            headers.select("SalesOrderID"),
            "SalesOrderID",
            "left_anti"
        ).count()
    ),
    (
        "Sales details without matching product",
        details.join(
            products.select("ProductID"),
            "ProductID",
            "left_anti"
        ).count()
    ),
    (
        "Orders without matching customer",
        headers.join(
            customers.select("CustomerID"),
            "CustomerID",
            "left_anti"
        ).count()
    ),
    (
        "Sales details with null SalesOrderID",
        details.filter(F.col("SalesOrderID").isNull()).count()
    ),
    (
        "Sales details with null ProductID",
        details.filter(F.col("ProductID").isNull()).count()
    )
]

for check_name, result in checks:
    print(f"{check_name}: {result}")

print(f"\nTotal sales detail rows: {detail_count}")

Sales details without matching order: 0
Sales details without matching product: 0
Orders without matching customer: 0
Sales details with null SalesOrderID: 0
Sales details with null ProductID: 0

Total sales detail rows: 121317


In [0]:
headers.select(
    F.min("OrderDate").alias("FirstOrderDate"),
    F.max("OrderDate").alias("LastOrderDate"),
    F.countDistinct(
        F.to_date("OrderDate")
    ).alias("DistinctOrderDates")
).show(truncate=False)

+-------------------+-------------------+------------------+
|FirstOrderDate     |LastOrderDate      |DistinctOrderDates|
+-------------------+-------------------+------------------+
|2011-05-31 00:00:00|2014-06-30 00:00:00|1124              |
+-------------------+-------------------+------------------+



In [0]:
print(
    "Orders with null OrderDate:",
    headers.filter(F.col("OrderDate").isNull()).count()
)

Orders with null OrderDate: 0


In [0]:
spark.sql("""
CREATE SCHEMA IF NOT EXISTS adb_adventureworks_dev.gold
""")

print("Gold schema is ready.")

Gold schema is ready.


In [0]:
from pyspark.sql import functions as F

date_range = headers.select(
    F.to_date(F.min("OrderDate")).alias("StartDate"),
    F.to_date(F.max("OrderDate")).alias("EndDate")
).first()

dim_date = (
    spark.range(1)
    .select(
        F.explode(
            F.sequence(
                F.lit(date_range["StartDate"]),
                F.lit(date_range["EndDate"]),
                F.expr("INTERVAL 1 DAY")
            )
        ).alias("FullDate")
    )
    .select(
        F.date_format("FullDate", "yyyyMMdd")
         .cast("int").alias("DateKey"),
        F.col("FullDate"),
        F.dayofmonth("FullDate").alias("Day"),
        F.month("FullDate").alias("Month"),
        F.date_format("FullDate", "MMMM").alias("MonthName"),
        F.quarter("FullDate").alias("Quarter"),
        F.year("FullDate").alias("Year"),
        F.dayofweek("FullDate").alias("DayOfWeek"),
        F.date_format("FullDate", "EEEE").alias("DayName"),
        (F.dayofweek("FullDate").isin(1, 7))
         .alias("WeekendFlag")
    )
)

(
    dim_date.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("adb_adventureworks_dev.gold.dim_date")
)

print(
    "DimDate rows:",
    spark.table("adb_adventureworks_dev.gold.dim_date").count()
)

display(
    spark.table("adb_adventureworks_dev.gold.dim_date")
    .orderBy("FullDate")
    .limit(10)
)

DimDate rows: 1127


DateKey,FullDate,Day,Month,MonthName,Quarter,Year,DayOfWeek,DayName,WeekendFlag
20110531,2011-05-31,31,5,May,2,2011,3,Tuesday,false
20110601,2011-06-01,1,6,June,2,2011,4,Wednesday,false
20110602,2011-06-02,2,6,June,2,2011,5,Thursday,false
20110603,2011-06-03,3,6,June,2,2011,6,Friday,false
20110604,2011-06-04,4,6,June,2,2011,7,Saturday,true
20110605,2011-06-05,5,6,June,2,2011,1,Sunday,true
20110606,2011-06-06,6,6,June,2,2011,2,Monday,false
20110607,2011-06-07,7,6,June,2,2011,3,Tuesday,false
20110608,2011-06-08,8,6,June,2,2011,4,Wednesday,false
20110609,2011-06-09,9,6,June,2,2011,5,Thursday,false


In [0]:
dim_date_check = spark.table(
    "adb_adventureworks_dev.gold.dim_date"
)

print("Total rows:", dim_date_check.count())

print(
    "Distinct DateKeys:",
    dim_date_check.select("DateKey").distinct().count()
)

print(
    "Duplicate DateKeys:",
    dim_date_check.count()
    - dim_date_check.select("DateKey").distinct().count()
)

print(
    "Null DateKeys:",
    dim_date_check.filter(F.col("DateKey").isNull()).count()
)

Total rows: 1127
Distinct DateKeys: 1127
Duplicate DateKeys: 0
Null DateKeys: 0


In [0]:
customer_check = customers.agg(
    F.count("*").alias("TotalRows"),
    F.countDistinct("CustomerID").alias("DistinctCustomerIDs"),
    F.sum(
        F.when(F.col("CustomerID").isNull(), 1).otherwise(0)
    ).alias("NullCustomerIDs")
)

customer_check.show()

+---------+-------------------+---------------+
|TotalRows|DistinctCustomerIDs|NullCustomerIDs|
+---------+-------------------+---------------+
|    19820|              19820|              0|
+---------+-------------------+---------------+



In [0]:
customers.agg(
    F.sum(
        F.when(F.col("AccountNumber").isNull(), 1).otherwise(0)
    ).alias("NullAccountNumbers"),
    F.countDistinct("AccountNumber").alias("DistinctAccountNumbers"),
    F.count("*").alias("TotalRows")
).show()

+------------------+----------------------+---------+
|NullAccountNumbers|DistinctAccountNumbers|TotalRows|
+------------------+----------------------+---------+
|                 0|                 19820|    19820|
+------------------+----------------------+---------+



In [0]:
customers.agg(
    F.sum(
        F.when(
            F.col("PersonID").isNotNull()
            & F.col("StoreID").isNull(), 1
        ).otherwise(0)
    ).alias("PersonOnly"),

    F.sum(
        F.when(
            F.col("PersonID").isNull()
            & F.col("StoreID").isNotNull(), 1
        ).otherwise(0)
    ).alias("StoreOnly"),

    F.sum(
        F.when(
            F.col("PersonID").isNotNull()
            & F.col("StoreID").isNotNull(), 1
        ).otherwise(0)
    ).alias("PersonAndStore"),

    F.sum(
        F.when(
            F.col("PersonID").isNull()
            & F.col("StoreID").isNull(), 1
        ).otherwise(0)
    ).alias("Neither")
).show()

+----------+---------+--------------+-------+
|PersonOnly|StoreOnly|PersonAndStore|Neither|
+----------+---------+--------------+-------+
|     18484|      701|           635|      0|
+----------+---------+--------------+-------+



In [0]:
dim_customer = customers.select(
    "CustomerID",
    "AccountNumber",
    "PersonID",
    "StoreID",
    "TerritoryID",
    "ModifiedDate"
)

(
    dim_customer.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("adb_adventureworks_dev.gold.dim_customer")
)

print(
    "DimCustomer rows:",
    spark.table("adb_adventureworks_dev.gold.dim_customer").count()
)

display(
    spark.table("adb_adventureworks_dev.gold.dim_customer")
    .orderBy("CustomerID")
    .limit(10)
)

DimCustomer rows: 19820


CustomerID,AccountNumber,PersonID,StoreID,TerritoryID,ModifiedDate
1,AW00000001,null,934,1,2014-09-13T00:00:00Z
2,AW00000002,null,1028,1,2014-09-14T10:00:00Z
3,AW00000003,null,642,4,2014-09-14T10:00:00Z
4,AW00000004,null,932,4,2014-09-14T10:00:00Z
5,AW00000005,null,1026,4,2014-09-12T11:15:07.263Z
6,AW00000006,null,644,4,2014-09-12T11:15:07.263Z
7,AW00000007,null,930,1,2014-09-12T11:15:07.263Z
8,AW00000008,null,1024,5,2014-09-12T11:15:07.263Z
9,AW00000009,null,620,5,2014-09-12T11:15:07.263Z
10,AW00000010,null,928,6,2014-09-12T11:15:07.263Z


In [0]:
dim_customer_check = spark.table(
    "adb_adventureworks_dev.gold.dim_customer"
)

total_rows = dim_customer_check.count()

distinct_keys = (
    dim_customer_check
    .select("CustomerID")
    .distinct()
    .count()
)

print("Total rows:", total_rows)
print("Distinct CustomerIDs:", distinct_keys)
print("Duplicate CustomerIDs:", total_rows - distinct_keys)
print(
    "Null CustomerIDs:",
    dim_customer_check.filter(
        F.col("CustomerID").isNull()
    ).count()
)

Total rows: 19820
Distinct CustomerIDs: 19820
Duplicate CustomerIDs: 0
Null CustomerIDs: 0


In [0]:
products.agg(
    F.count("*").alias("TotalRows"),
    F.countDistinct("ProductID").alias("DistinctProductIDs"),
    F.countDistinct("ProductNumber").alias("DistinctProductNumbers"),
    F.sum(
        F.when(F.col("ProductID").isNull(), 1).otherwise(0)
    ).alias("NullProductIDs"),
    F.sum(
        F.when(F.col("ProductNumber").isNull(), 1).otherwise(0)
    ).alias("NullProductNumbers"),
    F.sum(
        F.when(F.col("Name").isNull(), 1).otherwise(0)
    ).alias("NullProductNames")
).show()

+---------+------------------+----------------------+--------------+------------------+----------------+
|TotalRows|DistinctProductIDs|DistinctProductNumbers|NullProductIDs|NullProductNumbers|NullProductNames|
+---------+------------------+----------------------+--------------+------------------+----------------+
|      504|               504|                   504|             0|                 0|               0|
+---------+------------------+----------------------+--------------+------------------+----------------+



In [0]:
products.agg(
    F.sum(
        F.when(F.col("ProductSubcategoryID").isNull(), 1).otherwise(0)
    ).alias("NullSubcategoryIDs"),
    F.sum(
        F.when(F.col("ProductModelID").isNull(), 1).otherwise(0)
    ).alias("NullModelIDs"),
    F.sum(
        F.when(F.col("ListPrice").isNull(), 1).otherwise(0)
    ).alias("NullListPrices"),
    F.sum(
        F.when(F.col("StandardCost").isNull(), 1).otherwise(0)
    ).alias("NullStandardCosts"),
    F.sum(
        F.when(F.col("Color").isNull(), 1).otherwise(0)
    ).alias("NullColors"),
    F.sum(
        F.when(F.col("ProductLine").isNull(), 1).otherwise(0)
    ).alias("NullProductLines")
).show()

+------------------+------------+--------------+-----------------+----------+----------------+
|NullSubcategoryIDs|NullModelIDs|NullListPrices|NullStandardCosts|NullColors|NullProductLines|
+------------------+------------+--------------+-----------------+----------+----------------+
|               209|         209|             0|                0|       248|             226|
+------------------+------------+--------------+-----------------+----------+----------------+



In [0]:
products.select(
    "ProductLine",
    "Class",
    "Style",
    "SizeUnitMeasureCode",
    "WeightUnitMeasureCode"
).distinct().orderBy(
    "ProductLine",
    "Class",
    "Style"
).show(100, truncate=False)

products.agg(
    F.min("SellStartDate").alias("EarliestSellStart"),
    F.max("SellStartDate").alias("LatestSellStart"),
    F.sum(
        F.when(F.col("SellEndDate").isNotNull(), 1).otherwise(0)
    ).alias("ProductsWithSellEndDate"),
    F.sum(
        F.when(F.col("DiscontinuedDate").isNotNull(), 1).otherwise(0)
    ).alias("DiscontinuedProducts")
).show()

+-----------+-----+-----+-------------------+---------------------+
|ProductLine|Class|Style|SizeUnitMeasureCode|WeightUnitMeasureCode|
+-----------+-----+-----+-------------------+---------------------+
|NULL       |NULL |NULL |NULL               |NULL                 |
|NULL       |NULL |NULL |NULL               |G                    |
|NULL       |H    |NULL |NULL               |G                    |
|NULL       |H    |NULL |NULL               |NULL                 |
|NULL       |L    |NULL |NULL               |NULL                 |
|NULL       |L    |NULL |NULL               |G                    |
|NULL       |M    |NULL |NULL               |G                    |
|NULL       |M    |NULL |NULL               |NULL                 |
|M          |NULL |NULL |NULL               |NULL                 |
|M          |NULL |U    |NULL               |NULL                 |
|M          |NULL |W    |NULL               |NULL                 |
|M          |H    |NULL |NULL               |G  

In [0]:
dim_product = products.select(
    "ProductID",
    "ProductNumber",
    "Name",
    "MakeFlag",
    "FinishedGoodsFlag",
    "Color",
    "SafetyStockLevel",
    "ReorderPoint",
    "StandardCost",
    "ListPrice",
    "Size",
    "SizeUnitMeasureCode",
    "WeightUnitMeasureCode",
    "Weight",
    "DaysToManufacture",
    "ProductLine",
    "Class",
    "Style",
    "ProductSubcategoryID",
    "ProductModelID",
    "SellStartDate",
    "SellEndDate",
    "DiscontinuedDate",
    "ModifiedDate"
)

(
    dim_product.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("adb_adventureworks_dev.gold.dim_product")
)

print(
    "DimProduct rows:",
    spark.table("adb_adventureworks_dev.gold.dim_product").count()
)

display(
    spark.table("adb_adventureworks_dev.gold.dim_product")
    .orderBy("ProductID")
    .limit(10)
)

DimProduct rows: 504


ProductID,ProductNumber,Name,MakeFlag,FinishedGoodsFlag,Color,SafetyStockLevel,ReorderPoint,StandardCost,ListPrice,Size,SizeUnitMeasureCode,WeightUnitMeasureCode,Weight,DaysToManufacture,ProductLine,Class,Style,ProductSubcategoryID,ProductModelID,SellStartDate,SellEndDate,DiscontinuedDate,ModifiedDate
1,AR-5381,Adjustable Race,false,false,null,1000,750,0.0000,0.0000,null,null,null,null,0,null,null,null,null,null,2008-04-30T00:00:00Z,null,null,2014-02-08T10:01:36.827Z
2,BA-8327,Bearing Ball,false,false,null,1000,750,0.0000,0.0000,null,null,null,null,0,null,null,null,null,null,2008-04-30T00:00:00Z,null,null,2014-02-08T10:01:36.827Z
3,BE-2349,BB Ball Bearing,true,false,null,800,600,0.0000,0.0000,null,null,null,null,1,null,null,null,null,null,2008-04-30T00:00:00Z,null,null,2014-02-08T10:01:36.827Z
4,BE-2908,Headset Ball Bearings,false,false,null,800,600,0.0000,0.0000,null,null,null,null,0,null,null,null,null,null,2008-04-30T00:00:00Z,null,null,2014-02-08T10:01:36.827Z
316,BL-2036,Blade,true,false,null,800,600,0.0000,0.0000,null,null,null,null,1,null,null,null,null,null,2008-04-30T00:00:00Z,null,null,2014-02-08T10:01:36.827Z
317,CA-5965,LL Crankarm,false,false,Black,500,375,0.0000,0.0000,null,null,null,null,0,null,L,null,null,null,2008-04-30T00:00:00Z,null,null,2014-02-08T10:01:36.827Z
318,CA-6738,ML Crankarm,false,false,Black,500,375,0.0000,0.0000,null,null,null,null,0,null,M,null,null,null,2008-04-30T00:00:00Z,null,null,2014-02-08T10:01:36.827Z
319,CA-7457,HL Crankarm,false,false,Black,500,375,0.0000,0.0000,null,null,null,null,0,null,null,null,null,null,2008-04-30T00:00:00Z,null,null,2014-02-08T10:01:36.827Z
320,CB-2903,Chainring Bolts,false,false,Silver,1000,750,0.0000,0.0000,null,null,null,null,0,null,null,null,null,null,2008-04-30T00:00:00Z,null,null,2014-02-08T10:01:36.827Z
321,CN-6137,Chainring Nut,false,false,Silver,1000,750,0.0000,0.0000,null,null,null,null,0,null,null,null,null,null,2008-04-30T00:00:00Z,null,null,2014-02-08T10:01:36.827Z


In [0]:
dim_product_check = spark.table(
    "adb_adventureworks_dev.gold.dim_product"
)

total_rows = dim_product_check.count()

distinct_keys = (
    dim_product_check
    .select("ProductID")
    .distinct()
    .count()
)

print("Total rows:", total_rows)
print("Distinct ProductIDs:", distinct_keys)
print("Duplicate ProductIDs:", total_rows - distinct_keys)

print(
    "Null ProductIDs:",
    dim_product_check.filter(
        F.col("ProductID").isNull()
    ).count()
)

Total rows: 504
Distinct ProductIDs: 504
Duplicate ProductIDs: 0
Null ProductIDs: 0


In [0]:
details.agg(
    F.count("*").alias("TotalLines"),
    F.sum("OrderQty").alias("TotalQuantity"),
    F.sum("LineTotal").alias("TotalLineSales"),
    F.sum("UnitPriceDiscount").alias("SumOfUnitPriceDiscount"),
    F.min("LineTotal").alias("MinimumLineTotal"),
    F.max("LineTotal").alias("MaximumLineTotal")
).show(truncate=False)

+----------+-------------+----------------+----------------------+----------------+----------------+
|TotalLines|TotalQuantity|TotalLineSales  |SumOfUnitPriceDiscount|MinimumLineTotal|MaximumLineTotal|
+----------+-------------+----------------+----------------------+----------------+----------------+
|121317    |274914       |109846381.399888|342.8500              |1.374000        |27893.619000    |
+----------+-------------+----------------+----------------------+----------------+----------------+



In [0]:
gold_dates = spark.table(
    "adb_adventureworks_dev.gold.dim_date"
)

unmatched_order_dates = (
    headers
    .select(F.to_date("OrderDate").alias("OrderDate"))
    .distinct()
    .join(
        gold_dates.select(
            F.col("FullDate").alias("OrderDate")
        ),
        on="OrderDate",
        how="left_anti"
    )
)

print(
    "Distinct order dates without a DimDate match:",
    unmatched_order_dates.count()
)

Distinct order dates without a DimDate match: 0


In [0]:
dim_date = spark.table(
    "adb_adventureworks_dev.gold.dim_date"
)

dim_customer = spark.table(
    "adb_adventureworks_dev.gold.dim_customer"
)

dim_product = spark.table(
    "adb_adventureworks_dev.gold.dim_product"
)

fact_sales_line = (
    details.alias("d")
    .join(
        headers.alias("h"),
        F.col("d.SalesOrderID") == F.col("h.SalesOrderID"),
        "inner"
    )
    .join(
        dim_customer.alias("c"),
        F.col("h.CustomerID") == F.col("c.CustomerID"),
        "inner"
    )
    .join(
        dim_product.alias("p"),
        F.col("d.ProductID") == F.col("p.ProductID"),
        "inner"
    )
    .join(
        dim_date.alias("dt"),
        F.to_date(F.col("h.OrderDate")) == F.col("dt.FullDate"),
        "inner"
    )
    .select(
        F.col("d.SalesOrderID"),
        F.col("d.SalesOrderDetailID"),
        F.col("dt.DateKey").alias("OrderDateKey"),
        F.col("c.CustomerID").alias("CustomerKey"),
        F.col("p.ProductID").alias("ProductKey"),
        F.col("d.OrderQty"),
        F.col("d.UnitPrice"),
        F.col("d.UnitPriceDiscount"),
        F.col("d.LineTotal"),
        F.col("d.ModifiedDate")
    )
)

print("FactSalesLine rows:", fact_sales_line.count())

display(
    fact_sales_line
    .orderBy("SalesOrderID", "SalesOrderDetailID")
    .limit(10)
)

FactSalesLine rows: 121317


SalesOrderID,SalesOrderDetailID,OrderDateKey,CustomerKey,ProductKey,OrderQty,UnitPrice,UnitPriceDiscount,LineTotal,ModifiedDate
43659,1,20110531,29825,776,1,2024.9940,0.0000,2024.994000,2011-05-31T00:00:00Z
43659,2,20110531,29825,777,3,2024.9940,0.0000,6074.982000,2011-05-31T00:00:00Z
43659,3,20110531,29825,778,1,2024.9940,0.0000,2024.994000,2011-05-31T00:00:00Z
43659,4,20110531,29825,771,1,2039.9940,0.0000,2039.994000,2011-05-31T00:00:00Z
43659,5,20110531,29825,772,1,2039.9940,0.0000,2039.994000,2011-05-31T00:00:00Z
43659,6,20110531,29825,773,2,2039.9940,0.0000,4079.988000,2011-05-31T00:00:00Z
43659,7,20110531,29825,774,1,2039.9940,0.0000,2039.994000,2011-05-31T00:00:00Z
43659,8,20110531,29825,714,3,28.8404,0.0000,86.521200,2011-05-31T00:00:00Z
43659,9,20110531,29825,716,1,28.8404,0.0000,28.840400,2011-05-31T00:00:00Z
43659,10,20110531,29825,709,6,5.7000,0.0000,34.200000,2011-05-31T00:00:00Z


In [0]:
fact_sales_line.agg(
    F.count("*").alias("TotalLines"),
    F.sum("OrderQty").alias("TotalQuantity"),
    F.sum("LineTotal").alias("TotalLineSales"),
    F.sum("UnitPriceDiscount").alias("SumOfUnitPriceDiscount"),
    F.min("LineTotal").alias("MinimumLineTotal"),
    F.max("LineTotal").alias("MaximumLineTotal")
).show(truncate=False)

+----------+-------------+----------------+----------------------+----------------+----------------+
|TotalLines|TotalQuantity|TotalLineSales  |SumOfUnitPriceDiscount|MinimumLineTotal|MaximumLineTotal|
+----------+-------------+----------------+----------------------+----------------+----------------+
|121317    |274914       |109846381.399888|342.8500              |1.374000        |27893.619000    |
+----------+-------------+----------------+----------------------+----------------+----------------+



In [0]:
(
    fact_sales_line.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("adb_adventureworks_dev.gold.fact_sales_line")
)

print("FactSalesLine saved successfully.")

FactSalesLine saved successfully.


In [0]:
saved_fact = spark.table(
    "adb_adventureworks_dev.gold.fact_sales_line"
)

saved_fact.agg(
    F.count("*").alias("TotalRows"),
    F.countDistinct(
        F.struct("SalesOrderID", "SalesOrderDetailID")
    ).alias("DistinctOrderLines"),
    F.sum("OrderQty").alias("TotalQuantity"),
    F.sum("LineTotal").alias("TotalLineSales")
).show(truncate=False)

+---------+------------------+-------------+----------------+
|TotalRows|DistinctOrderLines|TotalQuantity|TotalLineSales  |
+---------+------------------+-------------+----------------+
|121317   |121317            |274914       |109846381.399888|
+---------+------------------+-------------+----------------+



In [0]:
from pyspark.sql import functions as F

saved_fact.select(
    F.sum(F.col("OrderDateKey").isNull().cast("int")).alias("NullOrderDateKey"),
    F.sum(F.col("CustomerKey").isNull().cast("int")).alias("NullCustomerKey"),
    F.sum(F.col("ProductKey").isNull().cast("int")).alias("NullProductKey")
).show()

print("Rows with unmatched date keys:",
      saved_fact.join(
          dim_date,
          saved_fact.OrderDateKey == dim_date.DateKey,
          "left_anti"
      ).count())

print("Rows with unmatched customer keys:",
      saved_fact.join(
          dim_customer,
          saved_fact.CustomerKey == dim_customer.CustomerID,
          "left_anti"
      ).count())

print("Rows with unmatched product keys:",
      saved_fact.join(
          dim_product,
          saved_fact.ProductKey == dim_product.ProductID,
          "left_anti"
      ).count())

+----------------+---------------+--------------+
|NullOrderDateKey|NullCustomerKey|NullProductKey|
+----------------+---------------+--------------+
|               0|              0|             0|
+----------------+---------------+--------------+

Rows with unmatched date keys: 0
Rows with unmatched customer keys: 0
Rows with unmatched product keys: 0


In [0]:
headers.agg(
    F.count("*").alias("TotalRows"),
    F.countDistinct("SalesOrderID").alias("DistinctSalesOrders"),
    F.sum(F.col("SalesOrderID").isNull().cast("int")).alias("NullSalesOrderIDs"),
    F.sum(F.col("CustomerID").isNull().cast("int")).alias("NullCustomerIDs"),
    F.sum(F.col("OrderDate").isNull().cast("int")).alias("NullOrderDates"),
    F.sum("SubTotal").alias("TotalSubTotal"),
    F.sum("TaxAmt").alias("TotalTax"),
    F.sum("Freight").alias("TotalFreight"),
    F.sum("TotalDue").alias("TotalDue")
).show(truncate=False)

+---------+-------------------+-----------------+---------------+--------------+--------------+-------------+------------+--------------+
|TotalRows|DistinctSalesOrders|NullSalesOrderIDs|NullCustomerIDs|NullOrderDates|TotalSubTotal |TotalTax     |TotalFreight|TotalDue      |
+---------+-------------------+-----------------+---------------+--------------+--------------+-------------+------------+--------------+
|31465    |31465              |0                |0              |0             |109846381.4039|10186974.4602|3183430.2518|123216786.1159|
+---------+-------------------+-----------------+---------------+--------------+--------------+-------------+------------+--------------+



In [0]:
headers.agg(
    F.min("OrderDate").alias("MinOrderDate"),
    F.max("OrderDate").alias("MaxOrderDate"),
    F.min("DueDate").alias("MinDueDate"),
    F.max("DueDate").alias("MaxDueDate"),
    F.min("ShipDate").alias("MinShipDate"),
    F.max("ShipDate").alias("MaxShipDate"),
    F.sum(F.col("DueDate").isNull().cast("int")).alias("NullDueDates"),
    F.sum(F.col("ShipDate").isNull().cast("int")).alias("NullShipDates")
).show(truncate=False)

+-------------------+-------------------+-------------------+-------------------+-------------------+-------------------+------------+-------------+
|MinOrderDate       |MaxOrderDate       |MinDueDate         |MaxDueDate         |MinShipDate        |MaxShipDate        |NullDueDates|NullShipDates|
+-------------------+-------------------+-------------------+-------------------+-------------------+-------------------+------------+-------------+
|2011-05-31 00:00:00|2014-06-30 00:00:00|2011-06-12 00:00:00|2014-07-12 00:00:00|2011-06-07 00:00:00|2014-07-07 00:00:00|0           |0            |
+-------------------+-------------------+-------------------+-------------------+-------------------+-------------------+------------+-------------+



In [0]:
dim_date.select(
    F.min("FullDate").alias("MinDate"),
    F.max("FullDate").alias("MaxDate"),
    F.count("*").alias("TotalDates")
).show()

dim_date.printSchema()

+----------+----------+----------+
|   MinDate|   MaxDate|TotalDates|
+----------+----------+----------+
|2011-05-31|2014-06-30|      1127|
+----------+----------+----------+

root
 |-- DateKey: integer (nullable = true)
 |-- FullDate: date (nullable = true)
 |-- Day: integer (nullable = true)
 |-- Month: integer (nullable = true)
 |-- MonthName: string (nullable = true)
 |-- Quarter: integer (nullable = true)
 |-- Year: integer (nullable = true)
 |-- DayOfWeek: integer (nullable = true)
 |-- DayName: string (nullable = true)
 |-- WeekendFlag: boolean (nullable = true)



In [0]:
from pyspark.sql import functions as F

last_date = dim_date.agg(
    F.max("FullDate").alias("LastDate")
).first()["LastDate"]

new_dates = (
    spark.range(1)
    .select(
        F.explode(
            F.sequence(
                F.date_add(F.lit(last_date), 1),
                F.to_date(F.lit("2014-07-12"))
            )
        ).alias("FullDate")
    )
    .select(
        F.date_format("FullDate", "yyyyMMdd").cast("int").alias("DateKey"),
        "FullDate",
        F.dayofmonth("FullDate").alias("Day"),
        F.month("FullDate").alias("Month"),
        F.date_format("FullDate", "MMMM").alias("MonthName"),
        F.quarter("FullDate").alias("Quarter"),
        F.year("FullDate").alias("Year"),
        F.dayofweek("FullDate").alias("DayOfWeek"),
        F.date_format("FullDate", "EEEE").alias("DayName"),
        (F.dayofweek("FullDate").isin([1, 7])).alias("WeekendFlag")
    )
)

(
    new_dates.write
    .format("delta")
    .mode("append")
    .saveAsTable("adb_adventureworks_dev.gold.dim_date")
)

print("DimDate extended successfully.")

DimDate extended successfully.


In [0]:
dim_date = spark.table("adb_adventureworks_dev.gold.dim_date")

dim_date.agg(
    F.min("FullDate").alias("MinDate"),
    F.max("FullDate").alias("MaxDate"),
    F.count("*").alias("TotalDates"),
    F.countDistinct("DateKey").alias("DistinctDateKeys")
).show()

print(
    "Duplicate DateKeys:",
    dim_date.groupBy("DateKey")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(
    "Unmatched OrderDates:",
    headers.join(
        dim_date,
        F.to_date(headers.OrderDate) == dim_date.FullDate,
        "left_anti"
    ).count()
)

print(
    "Unmatched DueDates:",
    headers.join(
        dim_date,
        F.to_date(headers.DueDate) == dim_date.FullDate,
        "left_anti"
    ).count()
)

print(
    "Unmatched ShipDates:",
    headers.join(
        dim_date,
        F.to_date(headers.ShipDate) == dim_date.FullDate,
        "left_anti"
    ).count()
)

+----------+----------+----------+----------------+
|   MinDate|   MaxDate|TotalDates|DistinctDateKeys|
+----------+----------+----------+----------------+
|2011-05-31|2014-07-12|      1139|            1139|
+----------+----------+----------+----------------+

Duplicate DateKeys: 0
Unmatched OrderDates: 0
Unmatched DueDates: 0
Unmatched ShipDates: 0


In [0]:
fact_sales_order = (
    headers.alias("h")
    .join(
        dim_customer.alias("c"),
        F.col("h.CustomerID") == F.col("c.CustomerID"),
        "inner"
    )
    .join(
        dim_date.alias("od"),
        F.to_date(F.col("h.OrderDate")) == F.col("od.FullDate"),
        "inner"
    )
    .join(
        dim_date.alias("dd"),
        F.to_date(F.col("h.DueDate")) == F.col("dd.FullDate"),
        "inner"
    )
    .join(
        dim_date.alias("sd"),
        F.to_date(F.col("h.ShipDate")) == F.col("sd.FullDate"),
        "inner"
    )
    .select(
        F.col("h.SalesOrderID"),
        F.col("h.SalesOrderNumber"),
        F.col("od.DateKey").alias("OrderDateKey"),
        F.col("dd.DateKey").alias("DueDateKey"),
        F.col("sd.DateKey").alias("ShipDateKey"),
        F.col("c.CustomerID").alias("CustomerKey"),
        F.col("h.Status"),
        F.col("h.OnlineOrderFlag"),
        F.col("h.SubTotal"),
        F.col("h.TaxAmt"),
        F.col("h.Freight"),
        F.col("h.TotalDue"),
        F.col("h.ModifiedDate")
    )
)

print("FactSalesOrder rows:", fact_sales_order.count())

fact_sales_order.show(10, truncate=False)

FactSalesOrder rows: 31465
+------------+----------------+------------+----------+-----------+-----------+------+---------------+----------+---------+---------+----------+-------------------+
|SalesOrderID|SalesOrderNumber|OrderDateKey|DueDateKey|ShipDateKey|CustomerKey|Status|OnlineOrderFlag|SubTotal  |TaxAmt   |Freight  |TotalDue  |ModifiedDate       |
+------------+----------------+------------+----------+-----------+-----------+------+---------------+----------+---------+---------+----------+-------------------+
|43659       |SO43659         |20110531    |20110612  |20110607   |29825      |5     |false          |20565.6206|1971.5149|616.0984 |23153.2339|2011-06-07 00:00:00|
|43660       |SO43660         |20110531    |20110612  |20110607   |29672      |5     |false          |1294.2529 |124.2483 |38.8276  |1457.3288 |2011-06-07 00:00:00|
|43661       |SO43661         |20110531    |20110612  |20110607   |29734      |5     |false          |32726.4786|3153.7696|985.5530 |36865.8012|2011

In [0]:
fact_sales_order.agg(
    F.count("*").alias("TotalOrders"),
    F.countDistinct("SalesOrderID").alias("DistinctSalesOrders"),
    F.sum("SubTotal").alias("TotalSubTotal"),
    F.sum("TaxAmt").alias("TotalTax"),
    F.sum("Freight").alias("TotalFreight"),
    F.sum("TotalDue").alias("TotalDue")
).show(truncate=False)

+-----------+-------------------+--------------+-------------+------------+--------------+
|TotalOrders|DistinctSalesOrders|TotalSubTotal |TotalTax     |TotalFreight|TotalDue      |
+-----------+-------------------+--------------+-------------+------------+--------------+
|31465      |31465              |109846381.4039|10186974.4602|3183430.2518|123216786.1159|
+-----------+-------------------+--------------+-------------+------------+--------------+



In [0]:
(
    fact_sales_order.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("adb_adventureworks_dev.gold.fact_sales_order")
)

print("FactSalesOrder saved successfully.")

FactSalesOrder saved successfully.


In [0]:
saved_order_fact = spark.table(
    "adb_adventureworks_dev.gold.fact_sales_order"
)

saved_order_fact.agg(
    F.count("*").alias("TotalRows"),
    F.countDistinct("SalesOrderID").alias("DistinctSalesOrders"),
    F.sum(F.col("SalesOrderID").isNull().cast("int")).alias("NullOrderIDs"),
    F.sum(F.col("CustomerKey").isNull().cast("int")).alias("NullCustomerKeys"),
    F.sum(F.col("OrderDateKey").isNull().cast("int")).alias("NullOrderDateKeys"),
    F.sum(F.col("DueDateKey").isNull().cast("int")).alias("NullDueDateKeys"),
    F.sum(F.col("ShipDateKey").isNull().cast("int")).alias("NullShipDateKeys")
).show(truncate=False)

print(
    "Duplicate SalesOrderIDs:",
    saved_order_fact.groupBy("SalesOrderID")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(
    "Unmatched CustomerKeys:",
    saved_order_fact.join(
        dim_customer,
        saved_order_fact.CustomerKey == dim_customer.CustomerID,
        "left_anti"
    ).count()
)

for key in ["OrderDateKey", "DueDateKey", "ShipDateKey"]:
    unmatched = saved_order_fact.join(
        dim_date,
        saved_order_fact[key] == dim_date.DateKey,
        "left_anti"
    ).count()

    print(f"Unmatched {key}: {unmatched}")

+---------+-------------------+------------+----------------+-----------------+---------------+----------------+
|TotalRows|DistinctSalesOrders|NullOrderIDs|NullCustomerKeys|NullOrderDateKeys|NullDueDateKeys|NullShipDateKeys|
+---------+-------------------+------------+----------------+-----------------+---------------+----------------+
|31465    |31465              |0           |0               |0                |0              |0               |
+---------+-------------------+------------+----------------+-----------------+---------------+----------------+

Duplicate SalesOrderIDs: 0
Unmatched CustomerKeys: 0
Unmatched OrderDateKey: 0
Unmatched DueDateKey: 0
Unmatched ShipDateKey: 0


In [0]:
spark.sql("""
SHOW TABLES IN adb_adventureworks_dev.gold
""").show(truncate=False)

+--------+----------------+-----------+
|database|tableName       |isTemporary|
+--------+----------------+-----------+
|gold    |dim_customer    |false      |
|gold    |dim_date        |false      |
|gold    |dim_product     |false      |
|gold    |fact_sales_line |false      |
|gold    |fact_sales_order|false      |
+--------+----------------+-----------+



In [0]:
monthly_sales = (
    saved_order_fact.alias("f")
    .join(
        dim_date.alias("d"),
        F.col("f.OrderDateKey") == F.col("d.DateKey"),
        "inner"
    )
    .groupBy(
        F.col("d.Year"),
        F.col("d.Month"),
        F.col("d.MonthName")
    )
    .agg(
        F.countDistinct("f.SalesOrderID").alias("TotalOrders"),
        F.sum("f.SubTotal").alias("SalesRevenue"),
        F.sum("f.TaxAmt").alias("TotalTax"),
        F.sum("f.Freight").alias("TotalFreight"),
        F.sum("f.TotalDue").alias("TotalOrderValue")
    )
    .orderBy("Year", "Month")
)

monthly_sales.show(50, truncate=False)

+----+-----+---------+-----------+------------+-----------+------------+---------------+
|Year|Month|MonthName|TotalOrders|SalesRevenue|TotalTax   |TotalFreight|TotalOrderValue|
+----+-----+---------+-----------+------------+-----------+------------+---------------+
|2011|5    |May      |43         |503805.9169 |48163.8345 |15051.1984  |567020.9498    |
|2011|6    |June     |141        |458910.8248 |36712.8666 |11472.7776  |507096.4690    |
|2011|7    |July     |231        |2044600.0034|188634.5689|58948.3105  |2292182.8828   |
|2011|8    |August   |250        |2495816.7335|232197.6621|72561.7767  |2800576.1723   |
|2011|9    |September|157        |502073.8458 |40165.9085 |12551.8539  |554791.6082    |
|2011|10   |October  |327        |4588761.8160|432386.8225|135120.8906 |5156269.5291   |
|2011|11   |November |230        |737839.8214 |59027.1869 |18446.0069  |815313.0152    |
|2011|12   |December |228        |1309863.2511|116255.7246|36329.9229  |1462448.8986   |
|2012|1    |January  

In [0]:
monthly_sales.agg(
    F.sum("TotalOrders").alias("TotalOrders"),
    F.sum("SalesRevenue").alias("TotalSalesRevenue"),
    F.sum("TotalTax").alias("TotalTax"),
    F.sum("TotalFreight").alias("TotalFreight"),
    F.sum("TotalOrderValue").alias("TotalOrderValue")
).show(truncate=False)

+-----------+-----------------+-------------+------------+---------------+
|TotalOrders|TotalSalesRevenue|TotalTax     |TotalFreight|TotalOrderValue|
+-----------+-----------------+-------------+------------+---------------+
|31465      |109846381.4039   |10186974.4602|3183430.2518|123216786.1159 |
+-----------+-----------------+-------------+------------+---------------+



In [0]:
top_products = (
    saved_fact.alias("f")
    .join(
        dim_product.alias("p"),
        F.col("f.ProductKey") == F.col("p.ProductID"),
        "inner"
    )
    .groupBy(
        F.col("p.ProductID"),
        F.col("p.ProductNumber"),
        F.col("p.Name")
    )
    .agg(
        F.sum("f.OrderQty").alias("UnitsSold"),
        F.sum("f.LineTotal").alias("SalesRevenue"),
        F.countDistinct("f.SalesOrderID").alias("Orders")
    )
    .orderBy(F.desc("SalesRevenue"))
)

top_products.show(10, truncate=False)

+---------+-------------+-----------------------+---------+--------------+------+
|ProductID|ProductNumber|Name                   |UnitsSold|SalesRevenue  |Orders|
+---------+-------------+-----------------------+---------+--------------+------+
|782      |BK-M68B-38   |Mountain-200 Black, 38 |2977     |4400592.800400|1252  |
|783      |BK-M68B-42   |Mountain-200 Black, 42 |2664     |4009494.761841|1177  |
|779      |BK-M68S-38   |Mountain-200 Silver, 38|2394     |3693678.025272|1094  |
|780      |BK-M68S-42   |Mountain-200 Silver, 42|2234     |3438478.860423|1040  |
|781      |BK-M68S-46   |Mountain-200 Silver, 46|2216     |3434256.941928|1054  |
|784      |BK-M68B-46   |Mountain-200 Black, 46 |2111     |3309673.216908|1059  |
|793      |BK-R89B-44   |Road-250 Black, 44     |1642     |2516857.314918|705   |
|794      |BK-R89B-48   |Road-250 Black, 48     |1498     |2347655.953454|712   |
|795      |BK-R89B-52   |Road-250 Black, 52     |1245     |2012447.775000|667   |
|753      |BK-R9

In [0]:
top_customers = (
    saved_order_fact.alias("f")
    .join(
        dim_customer.alias("c"),
        F.col("f.CustomerKey") == F.col("c.CustomerID"),
        "inner"
    )
    .groupBy(
        F.col("c.CustomerID"),
        F.col("c.AccountNumber")
    )
    .agg(
        F.countDistinct("f.SalesOrderID").alias("TotalOrders"),
        F.sum("f.SubTotal").alias("SalesRevenue"),
        F.sum("f.TotalDue").alias("TotalOrderValue")
    )
    .orderBy(F.desc("SalesRevenue"))
)

top_customers.show(10, truncate=False)

+----------+-------------+-----------+------------+---------------+
|CustomerID|AccountNumber|TotalOrders|SalesRevenue|TotalOrderValue|
+----------+-------------+-----------+------------+---------------+
|29818     |AW00029818   |12         |877107.1923 |989184.0820    |
|29715     |AW00029715   |12         |853849.1795 |961675.8596    |
|29722     |AW00029722   |12         |841908.7707 |954021.9235    |
|30117     |AW00030117   |12         |816755.5763 |919801.8188    |
|29614     |AW00029614   |12         |799277.8950 |901346.8560    |
|29639     |AW00029639   |12         |787773.0438 |887090.4106    |
|29701     |AW00029701   |8          |746317.5293 |841866.5522    |
|29617     |AW00029617   |12         |740985.8338 |834475.9271    |
|29994     |AW00029994   |12         |730798.7139 |824331.7682    |
|29646     |AW00029646   |12         |727272.6493 |820383.5466    |
+----------+-------------+-----------+------------+---------------+
only showing top 10 rows


In [0]:
top_customers.agg(
    F.sum("TotalOrders").alias("TotalOrders"),
    F.sum("SalesRevenue").alias("TotalSalesRevenue"),
    F.sum("TotalOrderValue").alias("TotalOrderValue")
).show(truncate=False)

+-----------+-----------------+---------------+
|TotalOrders|TotalSalesRevenue|TotalOrderValue|
+-----------+-----------------+---------------+
|31465      |109846381.4039   |123216786.1159 |
+-----------+-----------------+---------------+



In [0]:
top_products.agg(
    F.sum("UnitsSold").alias("TotalUnitsSold"),
    F.sum("SalesRevenue").alias("TotalLineRevenue")
).show(truncate=False)

+--------------+----------------+
|TotalUnitsSold|TotalLineRevenue|
+--------------+----------------+
|274914        |109846381.399888|
+--------------+----------------+



In [0]:
for table_name in [
    "adb_adventureworks_dev.gold.fact_sales_line",
    "adb_adventureworks_dev.gold.fact_sales_order"
]:
    print(f"\nTable: {table_name}")

    spark.sql(
        f"DESCRIBE DETAIL {table_name}"
    ).select(
        "format",
        "numFiles",
        "sizeInBytes",
        "partitionColumns",
        "properties"
    ).show(truncate=False)


Table: adb_adventureworks_dev.gold.fact_sales_line
+------+--------+-----------+----------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|format|numFiles|sizeInBytes|partitionColumns|properties                                                                                                                                                                                                                                                                                                                                                 

In [0]:
monthly_sales.explain("formatted")

== Physical Plan ==
AdaptiveSparkPlan (24)
+- == Initial Plan ==
   PhotonResultStage (23)
   +- PhotonColumnarToRow (22)
      +- PhotonSort (21)
         +- PhotonShuffleExchangeSource (20)
            +- PhotonShuffleMapStage (19)
               +- PhotonShuffleExchangeSink (18)
                  +- PhotonGroupingAgg (17)
                     +- PhotonShuffleExchangeSource (16)
                        +- PhotonShuffleMapStage (15)
                           +- PhotonShuffleExchangeSink (14)
                              +- PhotonGroupingAgg (13)
                                 +- PhotonGroupingAgg (12)
                                    +- PhotonShuffleExchangeSource (11)
                                       +- PhotonShuffleMapStage (10)
                                          +- PhotonShuffleExchangeSink (9)
                                             +- PhotonGroupingAgg (8)
                                                +- PhotonProject (7)
                               

In [0]:
monthly_sales_optimized = (
    saved_order_fact.alias("f")
    .join(
        dim_date.alias("d"),
        F.col("f.OrderDateKey") == F.col("d.DateKey"),
        "inner"
    )
    .groupBy(
        F.col("d.Year"),
        F.col("d.Month"),
        F.col("d.MonthName")
    )
    .agg(
        F.count("*").alias("TotalOrders"),
        F.sum("f.SubTotal").alias("SalesRevenue"),
        F.sum("f.TaxAmt").alias("TotalTax"),
        F.sum("f.Freight").alias("TotalFreight"),
        F.sum("f.TotalDue").alias("TotalOrderValue")
    )
    .orderBy("Year", "Month")
)

monthly_sales_optimized.show(50, truncate=False)

+----+-----+---------+-----------+------------+-----------+------------+---------------+
|Year|Month|MonthName|TotalOrders|SalesRevenue|TotalTax   |TotalFreight|TotalOrderValue|
+----+-----+---------+-----------+------------+-----------+------------+---------------+
|2011|5    |May      |43         |503805.9169 |48163.8345 |15051.1984  |567020.9498    |
|2011|6    |June     |141        |458910.8248 |36712.8666 |11472.7776  |507096.4690    |
|2011|7    |July     |231        |2044600.0034|188634.5689|58948.3105  |2292182.8828   |
|2011|8    |August   |250        |2495816.7335|232197.6621|72561.7767  |2800576.1723   |
|2011|9    |September|157        |502073.8458 |40165.9085 |12551.8539  |554791.6082    |
|2011|10   |October  |327        |4588761.8160|432386.8225|135120.8906 |5156269.5291   |
|2011|11   |November |230        |737839.8214 |59027.1869 |18446.0069  |815313.0152    |
|2011|12   |December |228        |1309863.2511|116255.7246|36329.9229  |1462448.8986   |
|2012|1    |January  

In [0]:
monthly_sales_optimized.explain("formatted")

== Physical Plan ==
AdaptiveSparkPlan (19)
+- == Initial Plan ==
   PhotonResultStage (18)
   +- PhotonColumnarToRow (17)
      +- PhotonSort (16)
         +- PhotonShuffleExchangeSource (15)
            +- PhotonShuffleMapStage (14)
               +- PhotonShuffleExchangeSink (13)
                  +- PhotonGroupingAgg (12)
                     +- PhotonShuffleExchangeSource (11)
                        +- PhotonShuffleMapStage (10)
                           +- PhotonShuffleExchangeSink (9)
                              +- PhotonGroupingAgg (8)
                                 +- PhotonProject (7)
                                    +- PhotonBroadcastHashJoin Inner (6)
                                       :- PhotonScan parquet adb_adventureworks_dev.gold.fact_sales_order (1)
                                       +- PhotonShuffleExchangeSource (5)
                                          +- PhotonShuffleMapStage (4)
                                             +- PhotonShuffleExch

In [0]:
monthly_sales_optimized.count()

38

In [0]:
monthly_sales_optimized.count()

38

In [0]:
aqe_settings = [
    "spark.sql.adaptive.enabled",
    "spark.sql.adaptive.coalescePartitions.enabled",
    "spark.sql.adaptive.skewJoin.enabled",
    "spark.sql.adaptive.autoBroadcastJoinThreshold"
]

for setting in aqe_settings:
    try:
        value = spark.conf.get(setting)
        print(f"{setting} = {value}")
    except Exception as e:
        print(f"{setting} = Not available: {str(e).splitlines()[0]}")

spark.sql.adaptive.enabled = true
spark.sql.adaptive.coalescePartitions.enabled = true
spark.sql.adaptive.skewJoin.enabled = true
spark.sql.adaptive.autoBroadcastJoinThreshold = None


In [0]:
monthly_sales_optimized.collect()
monthly_sales_optimized.explain("formatted")

== Physical Plan ==
AdaptiveSparkPlan (19)
+- == Initial Plan ==
   PhotonResultStage (18)
   +- PhotonColumnarToRow (17)
      +- PhotonSort (16)
         +- PhotonShuffleExchangeSource (15)
            +- PhotonShuffleMapStage (14)
               +- PhotonShuffleExchangeSink (13)
                  +- PhotonGroupingAgg (12)
                     +- PhotonShuffleExchangeSource (11)
                        +- PhotonShuffleMapStage (10)
                           +- PhotonShuffleExchangeSink (9)
                              +- PhotonGroupingAgg (8)
                                 +- PhotonProject (7)
                                    +- PhotonBroadcastHashJoin Inner (6)
                                       :- PhotonScan parquet adb_adventureworks_dev.gold.fact_sales_order (1)
                                       +- PhotonShuffleExchangeSource (5)
                                          +- PhotonShuffleMapStage (4)
                                             +- PhotonShuffleExch

In [0]:
monthly_sales_optimized.collect()

[Row(Year=2011, Month=5, MonthName='May', TotalOrders=43, SalesRevenue=Decimal('503805.9169'), TotalTax=Decimal('48163.8345'), TotalFreight=Decimal('15051.1984'), TotalOrderValue=Decimal('567020.9498')),
 Row(Year=2011, Month=6, MonthName='June', TotalOrders=141, SalesRevenue=Decimal('458910.8248'), TotalTax=Decimal('36712.8666'), TotalFreight=Decimal('11472.7776'), TotalOrderValue=Decimal('507096.4690')),
 Row(Year=2011, Month=7, MonthName='July', TotalOrders=231, SalesRevenue=Decimal('2044600.0034'), TotalTax=Decimal('188634.5689'), TotalFreight=Decimal('58948.3105'), TotalOrderValue=Decimal('2292182.8828')),
 Row(Year=2011, Month=8, MonthName='August', TotalOrders=250, SalesRevenue=Decimal('2495816.7335'), TotalTax=Decimal('232197.6621'), TotalFreight=Decimal('72561.7767'), TotalOrderValue=Decimal('2800576.1723')),
 Row(Year=2011, Month=9, MonthName='September', TotalOrders=157, SalesRevenue=Decimal('502073.8458'), TotalTax=Decimal('40165.9085'), TotalFreight=Decimal('12551.8539'), 

In [0]:
headers = spark.table(
    "adb_adventureworks_dev.silver.salesorderheader"
)

details = spark.table(
    "adb_adventureworks_dev.silver.salesorderdetail"
)

sales_join = (
    headers.alias("h")
    .join(
        details.alias("d"),
        F.col("h.SalesOrderID") == F.col("d.SalesOrderID"),
        "inner"
    )
    .select(
        F.col("h.SalesOrderID"),
        F.col("h.SalesOrderNumber"),
        F.col("d.SalesOrderDetailID"),
        F.col("d.ProductID"),
        F.col("d.OrderQty"),
        F.col("d.LineTotal")
    )
)

sales_join.explain("formatted")

== Physical Plan ==
AdaptiveSparkPlan (10)
+- == Initial Plan ==
   PhotonResultStage (9)
   +- PhotonColumnarToRow (8)
      +- PhotonProject (7)
         +- PhotonBroadcastHashJoin Inner (6)
            :- PhotonShuffleExchangeSource (4)
            :  +- PhotonShuffleMapStage (3)
            :     +- PhotonShuffleExchangeSink (2)
            :        +- PhotonScan parquet adb_adventureworks_dev.silver.salesorderheader (1)
            +- PhotonScan parquet adb_adventureworks_dev.silver.salesorderdetail (5)


(1) PhotonScan parquet adb_adventureworks_dev.silver.salesorderheader
Output [2]: [SalesOrderID#1424, SalesOrderNumber#1431]
Location: PreparedDeltaFileIndex [abfss://unity-catalog-storage@dbstorage7xahujkjryzy4.dfs.core.windows.net/7405604737487716/__unitystorage/catalogs/5d8d9234-455a-4e08-9d0b-e9030e8d5e6d/tables/bb02bc03-f778-461a-880c-abba9acaf034]
ReadSchema: struct<SalesOrderID:int,SalesOrderNumber:string>
RequiredDataFilters: [isnotnull(SalesOrderID#1424)]

(2) PhotonShuf

In [0]:
header_df = spark.table(
    "adb_adventureworks_dev.silver.salesorderheader"
)

detail_df = spark.table(
    "adb_adventureworks_dev.silver.salesorderdetail"
)

sort_merge_join = (
    header_df.hint("merge")
    .join(
        detail_df.hint("merge"),
        on="SalesOrderID",
        how="inner"
    )
    .select(
        header_df.SalesOrderID,
        header_df.SalesOrderNumber,
        detail_df.SalesOrderDetailID,
        detail_df.ProductID,
        detail_df.OrderQty,
        detail_df.LineTotal
    )
)

sort_merge_join.explain("formatted")

== Physical Plan ==
AdaptiveSparkPlan (17)
+- == Initial Plan ==
   Project (16)
   +- SortMergeJoin Inner (15)
      :- ColumnarToRow (7)
      :  +- PhotonResultStage (6)
      :     +- PhotonSort (5)
      :        +- PhotonShuffleExchangeSource (4)
      :           +- PhotonShuffleMapStage (3)
      :              +- PhotonShuffleExchangeSink (2)
      :                 +- PhotonScan parquet adb_adventureworks_dev.silver.salesorderheader (1)
      +- ColumnarToRow (14)
         +- PhotonResultStage (13)
            +- PhotonSort (12)
               +- PhotonShuffleExchangeSource (11)
                  +- PhotonShuffleMapStage (10)
                     +- PhotonShuffleExchangeSink (9)
                        +- PhotonScan parquet adb_adventureworks_dev.silver.salesorderdetail (8)


(1) PhotonScan parquet adb_adventureworks_dev.silver.salesorderheader
Output [2]: [SalesOrderID#1699, SalesOrderNumber#1706]
Location: PreparedDeltaFileIndex [abfss://unity-catalog-storage@dbstorage7xahu

In [0]:
import time
from pyspark.sql import functions as F

header_df = spark.table(
    "adb_adventureworks_dev.silver.salesorderheader"
)

detail_df = spark.table(
    "adb_adventureworks_dev.silver.salesorderdetail"
)

# Broadcast Hash Join
broadcast_join = (
    header_df.hint("broadcast")
    .join(detail_df, "SalesOrderID", "inner")
)

# Sort-Merge Join
sort_merge_join = (
    header_df.hint("merge")
    .join(detail_df.hint("merge"), "SalesOrderID", "inner")
)

# Aggregate the joined data to ensure both queries perform the join.
def run_query(df):
    return df.agg(
        F.count("*").alias("JoinedRows"),
        F.sum("LineTotal").alias("TotalSales")
    ).collect()

# Warm up both query strategies.
run_query(broadcast_join)
run_query(sort_merge_join)

# Measure five executions of each strategy.
results = {}

for name, df in [
    ("Broadcast Hash Join", broadcast_join),
    ("Sort-Merge Join", sort_merge_join)
]:
    timings = []

    for _ in range(5):
        start = time.perf_counter()
        output = run_query(df)
        elapsed = time.perf_counter() - start
        timings.append(elapsed)

    results[name] = {
        "TimesSeconds": timings,
        "AverageSeconds": sum(timings) / len(timings),
        "Output": output
    }

for name, result in results.items():
    print(name)
    print("Execution times:", result["TimesSeconds"])
    print("Average:", round(result["AverageSeconds"], 4), "seconds")
    print("Output:", result["Output"])
    print()

Broadcast Hash Join
Execution times: [1.2009708289999708, 2.11853475099997, 1.5942775099999835, 1.3350513129998944, 1.248816232000081]
Average: 1.4995 seconds
Output: [Row(JoinedRows=121317, TotalSales=Decimal('109846381.399888'))]

Sort-Merge Join
Execution times: [2.041074080000044, 1.3449060800001007, 1.4014185589999215, 2.047055711999974, 2.6310131709999496]
Average: 1.8931 seconds
Output: [Row(JoinedRows=121317, TotalSales=Decimal('109846381.399888'))]

